# Crawl corpus ViBioMIR trên Kaggle — tự upload Hugging Face

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung engine với `crawl_domain_samples.ipynb`: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Nguồn mới: `links_corpus_vi_en_zh.parquet`, **1.220.598 URL**, cùng snapshot với domain samples.
ID gốc từ 583 đến 7.466.161 và không liên tục; số URL không phải ID cuối.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).

**Bản tự upload:** `crawl_corpus_kaggle_hf_upload.ipynb`.
Mục 1–9 chạy quy trình crawl/xuất; mục 10 tự upload output đã hoàn tất
lên [AncientWings_R2AI_STAGE3_ds](https://huggingface.co/datasets/AncientWings/AncientWings_R2AI_STAGE3_ds).
Trước Run All: mở **Add-ons > Secrets**, tạo secret **HF_TOKEN** với token Hugging Face
có quyền ghi vào dataset này và bật secret cho notebook. Token không nằm trong code/output.

PubMed dùng EFetch GET theo PMID; các website khác dùng Requests/Selenium.


## 1. Cấu hình — chỉnh ID tại đây

Để cứu run đã crawl xong nhưng audit/export lỗi, đặt POSTPROCESS_ONLY = True,
chỉnh đúng khoảng ID và RESUME_RUN_PATH tới thư mục run đầy đủ trong Kaggle Input
(hoặc giữ None nếu run vẫn ở session hiện tại). Chọn SOURCE_PARQUET nếu nguồn nằm trong Input.
Chạy các ô từ trên xuống: mục 6 tự bỏ qua crawl; mục 7 chạy lại audit/export.
Snapshot crawler/runner/probe, cấu hình crawl và manifest nguồn được giữ nguyên.
ZIP diagnostics chỉ có một số mẫu, không đủ để hậu xử lý toàn run.

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus_vi_en_zh.parquet`.
Run mới dùng nguồn 1.220.598 URL. Resume phải khớp nguồn; nâng cấp từ run nguồn cũ dùng `RECOVERY_RUN_PATH`.
`POSTPROCESS_ONLY` nhận diện hash nguồn cũ trong manifest để audit/export đúng snapshot của run đó.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
LOG_EVERY = 300          # Tiến độ mỗi 300 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
PUBMED_USE_EFETCH = True  # Mặc định: PubMed đi EFetch GET, ghép theo PMID
PUBMED_EFETCH_BATCH_SIZE = 100
PUBMED_EFETCH_WORKERS = 1
PUBMED_EFETCH_GAP_SECONDS = 0.5
PUBMED_EFETCH_ATTEMPTS = 3
PUBMED_EFETCH_CONNECT_TIMEOUT = 10.0
PUBMED_EFETCH_READ_TIMEOUT = 60.0
PUBMED_EFETCH_BUDGET = 180.0
PUBMED_SELENIUM_FIRST = True  # Chỉ dùng khi PUBMED_USE_EFETCH = False
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

POSTPROCESS_ONLY = False  # True: chỉ audit/export run cũ, bỏ qua toàn bộ crawl
SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus_vi_en_zh.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh; đổi khi muốn run mới

from pathlib import Path
import json
import os
import shutil
import sys

NCBI_EMAIL = os.environ.get('NCBI_EMAIL', '')  # Email liên hệ; có thể điền ở đây
NCBI_API_KEY = os.environ.get('NCBI_API_KEY', '')  # Tùy chọn; không ghi vào settings/log

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
SOURCE_FILENAME = 'links_corpus_vi_en_zh.parquet'
DATASET_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
LEGACY_SOURCE = {
    'filename': 'links_corpus.parquet',
    'revision': '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7',
    'sha256': 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971',
}

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if type(POSTPROCESS_ONLY) is not bool:
    raise ValueError('POSTPROCESS_ONLY phải là bool.')
if POSTPROCESS_ONLY and RECOVERY_RUN_PATH:
    raise ValueError('Hậu xử lý dùng RESUME_RUN_PATH hoặc run hiện tại, không dùng RECOVERY_RUN_PATH.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
source_tag = f'ids_{ID_FROM}_{ID_TO}_synced_vi_en_zh'
default_tag = source_tag + ('_efetch' if PUBMED_USE_EFETCH else '')
if POSTPROCESS_ONLY:
    if RESUME_RUN_PATH:
        default_tag = Path(RESUME_RUN_PATH).name
    elif not (WORKDIR / 'runs' / default_tag).exists():
        default_tag = source_tag if (WORKDIR / 'runs' / source_tag).exists() else f'ids_{ID_FROM}_{ID_TO}_synced'
tag = RUN_TAG or default_tag + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    previous_manifest = json.loads((previous / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and previous_manifest.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Run resume dùng nguồn khác. Nguồn 1.220.598 URL cần run mới; dùng RECOVERY_RUN_PATH để nâng cấp.')
    if not POSTPROCESS_ONLY and PUBMED_USE_EFETCH and (previous / 'corpus_config.json').exists():
        old_settings = json.loads((previous / 'corpus_config.json').read_text(encoding='utf-8')).get('settings', {})
        if old_settings.get('pubmed_use_efetch') is not True:
            raise ValueError('Run cũ chưa dùng EFetch. Chọn RUN_TAG mới và RECOVERY_RUN_PATH.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if not POSTPROCESS_ONLY and (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and saved.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Nguồn của run cũ khác bản 1.220.598 URL. Chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
    if POSTPROCESS_ONLY and saved.get('source_sha256') == LEGACY_SOURCE['sha256']:
        SOURCE_FILENAME = LEGACY_SOURCE['filename']
        DATASET_REVISION = LEGACY_SOURCE['revision']
        SOURCE_SHA256 = LEGACY_SOURCE['sha256']
        print('Hậu xử lý run cũ: dùng đúng snapshot nguồn gốc trong manifest.')
if POSTPROCESS_ONLY and not all((RUN_DIR / name).exists() for name in (
        'manifest.json', 'sample.jsonl', 'records', 'crawler_snapshot.py')):
    raise ValueError('Hậu xử lý cần thư mục run đầy đủ; ZIP diagnostics không chứa toàn bộ checkpoint/raw.')
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/{SOURCE_FILENAME}'
SOURCE_PATH = WORKDIR / 'data' / SOURCE_FILENAME
print(f'File nguồn: {SOURCE_FILENAME}; revision: {DATASET_REVISION}')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld']
if BROWSER_TEST_MODE != 'off' and not POSTPROCESS_ONLY:
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Snapshot crawler/runner/probe đang có được giữ nguyên để resume đúng phiên bản.
Auditor, exporter và diagnostics được cập nhật từ notebook; không thay cấu hình/hash crawl.


In [ ]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pmF-ESPpaewDu(MKhiS$lU#>EuJQ<npYObg|%*PLOm51H8rDaJP4N%=yxMh+LBkB=AF?0>p5Dzz7VSYy^g51aXi&Ne_9T;s1mAOY*Dg?wOwbBF}IHh}@m2uBxu@s&4M??tVU*L}4ODn3Qswhvj^4l;UGjig6fBle8$qQ6Yk4jQ^raqTobEksOO$z6s@-C}vWYVv^<}5TiUei$q=}-tO-1P8ertUWy=}W<g%aok^a?Vw7U>s0`Dj5E|fNT7h0};HN(cGxK>C6tgHislS$SB!Yr}`?44XnKZ9{T%?JjkOd{JG1#Bus{zWfi!`pk)r*UHVLnz#I7-KI9F&21bRI{ZEORNn$B~R7m{<5rY&?FYC{C(mRHkWE?<mSEe-w;nde2oJfgz8jU_kJ!$fB^^Q6LeB2y2&yb~X?4JUyeeuPewIRI}GRJ3C`J0sq0+rQLS=I|Bc5Syp+%lJKH57#IA~BQU>|=cOwXY%NTt1E(q{y}xog9gS&}##t~bU4m*ObA;plI1Llml8cw5c^pLHPo(QS`}CpH6}-;l{kS#KVivdr-bSc3>f!CZ?|q~IyqP>7hf`T#wb@&6=ChVb?(RuCo_B>|Un3k#W=ISKz4p}CdXHV<<T8tbk#sr?9!^A(mS&Y#lzEuB8kNQ0vow)dE90!3`^3dK9aW@shat;+zOsb)^hk?@Gnu-YM5oKh7+Ri>WL657L6t2%Cu_(bL%Vrbd>KTQ<j)&9suH%J;3SeueC_Ofzcx5G4$cc`W^lh#%MpG$hFGL+D<m>Vj(m+J7iYOF3YVNkfhn^&nU}I4m0J8LWsr91Q)esrtU8J1*oSfZIjH2YNnWH)B85T*a3foK3aQo@L3m{_6(0<Q^PG`A$Ptes`BIjSC8{8VYc*1I3HT|D!|Yvz1*`w@r?MQ)#E+hS<~e3p%oq5$SnvZ1W$Gb9B6HBtiPBL(u?x@+&{N!e7?%9VVSafXb39n)*by=Ulg=k;ditnkRP-;4Lo#KJU=k){cG;TXcZIX}{OOZN@HR(tZgKAwvV^>YDk@#V)D_3Wju&K^OvbK)Y3Wce)6N!D<FND_UFj8FVd)A1{Y#mLles9;Du+PJu#8}r@{P>rA~=Ee5J5ztVmqe`&J>7=$ygE}*lsiz=|q$>sWOGk1>1Da>B5qN367ePV+Ri$Y>4BT@?2-H@oO!#sVrSb?}89H9WopGw8c;+yV_DnxO-o`_)NrM5eG1mzPK>tfor(fRrL6sBzxIp+{{R=g)K}5S?CwAo~i(=WG^6Hzy}of4bxa{sMfZr5Rr{w?b6fN6{0i3R||JAq7Wo)yxRq$nPI^+esqI$?9>5My|DG+Ym81`yvi%N_%@OmSFi`Tg5+1L@`+^PY8iFJp}6<|Ba)hnKaWtFO{opF#nRdm4EgTzo@~*+A>Rxo*}RT)gzL6;&kK;@QDVSF4XzElk*WxS18DA!_lNvqq$45TXc&%T?n<QW(Dn?-wpiIl(+Zr5mq3Aoq9Lwf4W3(*GV4{Ar`YBv$l}1PX|xf)A_&a{LsLf3rBbun3sKi{9PcnA7GA9-O71!@s9cbwet}wy@I^s<G7h-7>O=8fQJuKC^HuR(&H-I<b4g}ZJ_@Vijm@ck)QTs(Elx3=+ZYeS?;)Gs<N~>z$eetoYf=P7&|_Xdq&TV6snZC*a})6KKtF*ZDK96Kl5K?qQRG@cOK3a@fifVY`W$&DUx%+{I*f1$v{TmwNLMhVE<7=uhKd<Os}UD~)DDLABc{b_Pc)8I?q6{%Z-*k%qGegjHr0aVblVjaPo`;3)g6PgUO%IEO0TB!?OLy&Me#$utPRv6Fb<KFgSpRy*gokucIMVJVQTefD%rkx_Tc5qCx7$o@gr0HIBlH97)-5lp;B6(cMevV2EOXeIY|`zh5{a&kKGp1m4A-X7Z#`wj5wKA!BpazR)`jqe#~OPk8++Z!@QzL{C?5i40$-%UhOSfmSMK)w&dyMx~}QNG*RsJ1s9ZUnHv!Tfc@LffQ-Ap%#Suyih6CMUS-Zmgesej5?NVQJGR-DqEq{Y176*UBisvvf6?Kc)#R*A(dKv7W7nR{yHqemndZ`}nVDNd);EUbw2CmwscId^K>n%$xrz#sx!Yo(j*`HM62hvwRFG^%QhA;MnMCMgh+%s#8oh<+RL&Vey$%nC94qjFLWQgCf7$T6wME;OY^-%DF7RlEiFRvBgdDA(mPHejZx{8KtNr3O89=oSzKLjGC0`{vs%0v)a<^sl%@%DOX*G>m$hIW2Mor&pSG4XnJHjZ#uNMtixrI@^VV$=|Jy0WJdm7O~K|Rx-QnPN^k_HxMVL3~yvfksyEX_r#C0(mpr9P)FGgYk$;hKVd#qauU@7fZf4GvovSZ;q^=uR~bU@vBRV2w6Bo<8`f3W8i`t<~;q9T2ylT;!=CrS~A8a)GJFGnnNRBPf#gFz{*S;`{bWFBoI*dWq}wdJr;9E|;&X5RNq<A#H_ML)0G4OsHss`wl`XFmy9NxVyi<b-hv*md<394V*8veoT}8T#pTN$%6#E;AtZUxoI@U#tRI6xFXlp^t4dXpGfg9B35W&kpT^Op6b$558q8ooP-lh?>H7?CP{#M=w|g}k0z2$w@#BI15yNWhDweAB4=slM1E~h9BW8ljSZdQw$aZ^2IWa(?ZTQo`GMF#JVp9i9g)(+6*ee?xFM+)QiW&DTB*ZP?yjfldKsr3JTVbrXy>AfTsfc`I`JaLy=c-Cf8Q3soJVz6su_FosL=bZ<2|NZniN_IJx^I7YR73BwY0RYEecC4r6}ub@@uVf4qzIU)|s^jHS7iYrD<@JxL8pZD@r>~ezIO8CeNL~%ui`YR3y8qf$D2$wl4SX{=NI+JA!_pygPB6HD#J&&#N-DRv11L&Q__V-&<4bC0lLndw9ci!Fk#C)pd5Og%96+t>^hP2X-w1b;Tr%u<2rO?2?`lnz=lPLhxtIpwqz+txWxF&X3z=5#y@uu3kbK-)cL&ZeiCkw{XsJDpdo_yIcxaE0ft{b=1*m3WWlPBXV5F5e*72o<`}3>wHJY9MceT`E!F4XdLzYMK7}UMWP_tXvkGrZQJ;Ziuy&Gv`4aJKcZtc<g)^ed`$tadmzq&P@kwpkwD5Coo1+$O5@a{ppaI7sf~d0$nq51+E&9FEhjjKM^0KGbij~DP9qmQSgJqpZhc(>&`A_b3F3ITupLDeL(BzNHx&Bwx<ukwD|MvTk8kTVlU-_;+rHAT)oTylXuzn$kzGe&H5C>pWtcNEMGZ!E1$oqRWLU$)Q#~TZl6n|;6BnD%t78><3lUHzwNGvk!L+FZca34#S?NE`dL>c)ZH+)TZ0H3xatoQZcO(;ZBGSlpLI*5s+DqF^k2TIA6*};Nhzxi26ha?Bte_$bHqlwHZv)9l*VlY92)(0eq>SR8IDh&4(Wkw2K_7A^sjDsFNGw7RJoZgJpoZ+aC|Frb2YOvqH~HiWT77E{FjqIDT6ev$zqU%I{T^!{8=5}5S=3p(Xsx`d^Kwba`StsMPNsXOv#bAx{i>7cKm6>EzrTL}>ydc$`O^b&E@Lb9h&NZ?u$fP<-~Vjnbyj6SZTx!nvH4I1EXiSsiz7Y*<t*)t!#8S9>*$LIJqb%l^InF#2OK?IJ*R?FibAWxsSlioYpH_iJpFc5aY&nIe>z2KENJ*`PSf+1Hgq$i*0TD@U3wf5HG8_3<ZY<EZNR6jMXR47{m~2xc1Zh_uK3xuwQ3Qa2+RD99P>iOB~43tmynx!!qc1joO_LND=><&>Jz!8LDS#Z6s|RS%@{EFlPZel7K@!t<dyMlCe)lDC`fadaie6p0)ujJccqrBnM|Kkcy=9cZ;zoiX?|5(G3Gco;tb>IHjHRCPTB4QWq&gaRrI(!u*c`h4Fcr^E!dkR5R@f4X=*`zI;mw=k1M875e3BopX}jaI><*eI=ArLY&^L{3w{3a<#|+`f823D?*GN*@yB<D9XidRw%L2Kg}5$M*>dQR@9y{Z*{(K=u)lRIyurp80ehZq;|o6><-Y3Foy~H;Bc3!T-MSNL=2_f#QvgM9#ps=m&BatEGH<$aJ{M>2;wkb*LL3GnTa}G%sA<I|eJbaXqz~aF4`Tj=yIlNwUHWT>a&MfcnMz%oH@q>!)RVokH3ln}txug|@YuXP>brez*`DcssM6c$>c24QplfH~;*;xlZ_#l)4Z;I)8eYHuDFmMfO>QGKq{7r#cBa##%ayR+D)KCge;SSurPE3Igs!(GvV44=MVPndmvNEylz!+_sXiWl@YOMlC!Aa!@`IzxLsUUW`Yca-M_UT(EjXP_LdYY{{W!=^tIUT@ojU7k(Ul3~2b)jD+qCpJehH6y%M<s`&f0~LpKjf7W5sQ~T3S+{lh7C4_>fmqXwx?a-3i;(i1P_IMf7#TUmLWlNvdK9W6ph#G6f!DS`(Ho6#}X`GjB|vhgaIhV(>&4G?wR}%eKjtH)4~DBUDi%imVP#n+uk{>7d3TN&vOd)tRYdE`1d~m9r|lV-a@ej+82vb9bk8(%cr;EeTlIZYD)L`_^Ralv=cjtjV@rvELC#V*h3`t21C*2^N)gBkr#FG(g<x+?veF417DWbw^eEx1d&xQa+M&9#mx-bNYa_?IEw&7j_R==`>fME47&R3f=M21r6=5p0kR5R$-2%lbvrn#J=k6&BR4@Yuc^!w8cBa^<kAeU7n1Nt2irM4MhD<QRNsZ7o#v7uyDJ4J1i*&_d3n}&U*H`hM<ZJW@Ky8Ge_hW-FzKIK4wIMY6c`)&Kt<rM<{jRa`Tnp%C?Z&I<x^99PJJC&5pbtznFm+OC9%p3+^E4va<Jrf4TSXq5I%S<r-{Vbm@s2&gzwPjL=n8qlW3|sH$#h4@5mLV|bh{Xj$5aQVfd-j}c*EpJGGSF&CCMkB-zml6qpKzET{mXBYj|dC0mWBE;kzLdIM`oz?5hu*NVGoL~6+K1Q1Ntd~?n(a3$}-8uqrMGVg)KdIsqj89v^P9s#(l6mt4V&XvMI1AetYV|cmtP<v*#ZX!P8x1v6NB76h;RlGWbdtmulaCJh(mWj!4|d;#yGMsHCMJSALRk)W83mrQijE$TlOfOMSydeFsb@<;!qL?UjWjMM=uP`~Kg!PO3O(5CRp9}A9rK|6z5Chu0coK(li_p*Fz=&<p4MEPz-wgQze@|#2zNg3j7Fn_45wSzSO4xkEY)b}Ghz=;qr&_`--hVp&lJW7^rHuwcv8w9{;r^klcIljk_-Glm<AcMuz)@<&SsEZk57pE_})9qgX|zhbe=@%8Gq~J^!5z*7UY52(tUeN3y-q%1xTcM|IS~3^x@)AmDj7XncY2lJ`1nk{ZC~Vm+&)DUVRgyGrE5N_f;KK4)*|mm>oU6e)qddBvbWk7GC`uofjkM+`s3Ehu80afkWy1>Mhn|3d3b+dHw!()JmXQz4{gIWD|0D60ZL+bT+$w_iF{Nl<A2-{NvTH=Fs~ee;=6Fr{J?hGQbzSxPJGqF)ja108VHU^ve(<p{uvaOiaV8x8C7icH|6~hs(9eH*o?-rgXH^{EddE|5T&71~aX%i%fP8wNV@$>O^r=j~vDS{_Q`B3#@kY)%$6r1ib(vR8Z!o@Pe2<#E3`m0B7Kk`8c4UQZ1a~!y}R&Zcbdk|2f7zjTpOHMra0p9t%Aa>OfN;f=n(IA-m;Z&YD>GBN;c%gkqDHfmuV_X5t}h2jeD1hc5QioGcSvohPm0-ou{hrxxjM6C3LG09Yw5ETvZ~Bhc%m<l5ZT0H%#0C?0bFx+8wrn0~7Nx=5L!zPUKr?FoI``gC^v{$Dbq2#lNU!h5lIVO8i0Z`XD=rzdEV`Np_JJ-)OY@8u63^f3Ky7dk}!v6lA~u>XOjFngSWR;0CGWP;;`3K!x+8`<$qweb+#ZiOX-W=ExdR4cr_CF1N8191YkMDvP;`1ylZxR25ZX%n4xAw68IqBqfr%c)x2wHJ35YXjARb_(LF4h|}0fCTB(R?Nezw<E-utKVj}E|RNn=Hi@QC0O|HxCoge1SbJaM@WfS@D?XJL^u&Khp_@aBqEHesaMy%JnZGS_>IzMMtXbqo4HnBf+ZoYXyYYB_n-L1Wp4)mhtU?Ev%=>k#nxYnOYxHE*!#XD;7if#^#uRy<H=ClxaoZ-0JuOkw*bfmcP9AcnCn|z)va1*m8z-pIE%ZcE^E*olVfo7_<E(({hGJ<Qx@ZsM+f5TC2uwF>E}O;>Ztan4*!Vj?<kNbv!FTv7<T!0i5=YuHf`Zmgq75W?O5^Pi$_mh`7a(n|NO-(FCK4-74dj;v4TJKxkZn_*JtQ2j8K{Seu6QU?{h)z`*ah?_w|ednHmG6G1&7cbR?5-3MB4^D8+Y(cK#2Gdrm$',
 'corpus_runner_snapshot.py': 'c-rke%W~Yvk@x<J8u|dV+zpbWhrN_>pe1=`Vl<+bqGn>Fw*y8us=HyL0SpQtMN|9-`{3|(9TN^m?1saRJxYf?$l?FspKNA5fT{w}-PDXXd{|Ayuvw4HtgL)hW;XZs_D-vmlq*WuDlBM3qHG<;Dang$MhQ)qaY`8p(}?8NEQ#5QeDQRg7nITBB2JfNo)vk;$b3cT=Xn;VCG+<7_6G4f&x(?S#WD{IMhA-`Tf^EUq4P4%Qby#0Cs~!2v>1_yF2X7)qj+Ab&2x64K8hbnJoBnDPU_K2efky4QuTQqmMeiThflCa;d+S^@-w?)gTdF&PG5a-@`Cup5yu!8RhrUbeDTm3Jbm_y<JZs6Ucv13fWSYef-?u<lBT6|M4Yd(U&l!j?mzSni2F^PM%g7JFV4u}fp_qjz`&yikI9=y4~FD8&lCC$oqZXX`w#E`#Jm59xL<yL_SN$dN#b)#J_FWeL-J$=G}HaV`v3s{Bd@|mSj2LRGZI8ZGzqT&|DiV`PFR+7otF%LzYo8b@tS58QaAbZ0OlV&^rkR8tD+^v(Zip@r~m^On8Q^BaLXCr|BEnTR88{1Aj|nx!Q{0XvJlGAc?MhqU=9AmhZ0f8Keel%VWe%jj}$j^WpNeMg#U{^w+Q2;DriuaiP&}{$(|K(1#m^{ETVWrxR6FcmL^xusF^md@kN$Wn45Fn2eUX0iz_uOpn}-10ScoDP2*}EEaHOU&a<MjmH^;@E@(Mlbxzk)*Ul`2PXUX6jdUI!8~_EES#eGaWb7gQwhVK?2z-rj#CvGpbq{ZK2{=TaJvwMhfX%KXGwuO|Mq}1x7?&C)hsLBv(x;=r&0sJ9EhhY4S-1wFXYRZRFB8;n06sk8h}U5nFDNTPvE;1BNkAQTVHw#cj#}g4pflwA;zhlF0T&yrLbg)t1v<Vuafaj*zs53=a{-vQPai&VrbjjJiZEuBd>tkgeO44%;X1GQQi71dqJ`;_MxMle8J8>4jn2z*nz}TdXAxLL->J&Q_@_Wc$OtQe8|wz8WD6X9f}8olnVW!(9g;;>5d1{I`igycWE}C}*En~FQ^f}mMfbXxIC11m`M!d062Ncx`EiLH7;ZZ9(<)Emd00}p9n3pZX9g{-iW;tdy#+`_5gE5T0z#4qe-Y~?R<yxzT){Z80zd3si&f$bH=LDw>Ue2Z@_inC6SI=J`T+z<sGpKVB&1W65JD|Eu2|@S8n}pZI3oH;Rlq%RzPUPG>MRq!V7iHs3+GAg0)dD7uTP(ob<EbN>7epR%#F!bfnRb}I@cCsxf_=JHTW9lF`T;pGL+R3lgz2ND~=$5m~mbsH+2?EmZC3qVcCd;^EqWKU}acV%y+&%J_(+_K6&CuID&^_7UbRse@fq!MK~{M<PZvug*ab+e)8h8px!yG(W#5OEaH+HMnI9?D=+E}hqib~){2pDD`9Ll@JcRX{oGPufMYhdOo5cPaWg(;K)0dmU@SY2^Stekc!Z{KZz7VAACXg_0%Nx@!2u}*A%tg01KgRo6D_9mgvfCz7|Eb;1oIiADV&rGsV9X5U;`lP$|H;vRl`ABxC8`!#;fuOjFD?foQc^N+XQpOkfY{?U9PddUJ0&~yQ5~=sx0%)wc3QU852dL5S&9;XhLYl4UBxaW91esf@c957k{y5OYwe}&eS3%uf^7Tbpc95B^%_gHp|p1onyw*tsK*IMsmT}ued-=l~G5w6shyXeOMa}3X^0O&d)I-g$=&T^`v2uprl3x0#21Ono8Ig@%H}Xrcr2*0&fwc)Mz-YFHQ)$f#2!7MP;!yb8Mf>ur38n!9mwnTP~caMbkcBLVQ?4Kpm7<IrW_q(Cx3vb@CXaD0u#rzKDtXwkydsH9(Dma!;nHEj<3W)`<5@uV?|`1soef)t1)F@&3K=(#jb(wLJXw;pWrGHI>v{0+b#M$&XD2cdZ$b`|?76F!$FLRS2NcmKGFWzMDb|6)EOfUL~P;r*8<-SymNu8UP%O$7QHKalkm2@FawHJqXM|Xj)n2n;_N31{Vr5Ai?q2ZR@H!P!wW&eIqym%fT<;Iv#OPjVuF1-Jn){Ax6+L`fbAKj(})m1kvTmwRl!6I2zbm6Q>nzFlCpRZaAk`M|yY)zl-Mqd_W3a0}6@+P=FCC{1E9I{A9RL9H?ET(~Rl%+0+B)+bT@r@=61sMixo96cD*}Rv(00)59BXD)soP8&j9#HMW`H&0!mu>9}-qEa-;UakPk8Q7wOm09V^32lF*PB5w3s25=L_<0(ER9Qoqsu^E>ETw&2S<+FUoyf6Wx3AJ$<mwZ!(U}ulZ?$Q@$WI0<tH<1@P9qN)eaR6!v3jz==q0m5-uUiK$oysyv$(y2m)8Q~LgoqU_XLdPkF>K?Z&7%rUp4j-NE%+QMLt_UD<0s*I7KNlZBJQ~8JQNsGoqYMQmIgX6;tb4aEMD!`YTc1cl%mSDARe`tq!u4^dmELj0uU^S(*W|200iIgOAyDrXOmzEl7jY3rc&La9cdCw5j0-BNf`*}njAO(aE+pxFT?c$qX%AP%~U?3VbDkOT(X)~>ow%x&;;_(h6gN1Mk)hyb?TWSr|`CM9AyArHP!|)))v{vl+qW^5=_Jadbemxq;IHU&AJ|ldPI7`L#Y~YZ5xcxP3Zp?v^z(Z99Vb2`bj%r-T5?d$7_)p{XaX)hLiXoaS}TT+jS0QueZSx$|`|0O3L@ys&auTBG46-P^siX>X{VOMt!6j_D6(@twK#i>@!kYvn`ygDoFCu6!`T<QlJ266o_yrrb)!DU0-O?*5~gZZ`>i`s3U>eb#IS2`l1Y*h8<&$kB>73-GaG$)G@7N)-+s00FRe}0yk-1Ou+@K6GR@A(VS6_D64<2bH2*bWpEK@K~!aPxoJ|Ey1>2m7N0#GYAJS^$M&IA=e>d?cDnrH#7*fXX0?4s%K~zF#jF`1_MU<}`-Ts=0>DVfB%YV~LOkuQLgH5tp~J53Nj!mz+-fHu!!ldP^I&#Wg02dM5zH0%(F}1#cHyakd4B4OEdQ9rOGkRIgjY&YYkowYeE7qhT+$f>v}FF_zf+RjzW;qp%A&d=^V@fS&hb`lVQ~Kbmx`3}hyUaS*^HF8?|%nrI!q<cZ{PnZB>1#Kx&RK`7LjTQ{yDyV_uZOAx9|RxdR>qgdw=`QhwlLD?fXBI>)IjS>|g7?@&-{oCKn(6BO`zN?ce`;`~JH*xt8{2qC3m!&0~3@W?EWBc%9lQ7w4=#b^{l%Kz88v-5*Pm@F7e{0)`wDkO@q#!kC=jzWbk&d|ScsNP&!5LQR#f$-llne*VSTzXhkyUY?wudF!b2)XrP-R8+IylCy@Fl5H@Ezp0BshC<i{qtE%ssnjL%wgc>VJSP1AN5&6W+h`{SNrYF@m3eLSg3F4ETNit8$u+KS0-_tz47Fn?ZMGwB6w|!C6jke-xniXZ7N?jgWmrEA**uPYUiXEYf4iqk@sClAk)=;oz?+!g!slmaFGrGH>)UsK0Dj>ezLItZ)4?dQ|0TG5M1I*eDZdogG#zZwJ?EQeKM$Uro}Qe(g1f~%|DHPDuOJo8Yh}#RLZyMKUL_<_)vKzsJLdtf??t>Ek$WQ|i?9BJ5!4Oe1Yn?iQ5cp{9JnN&>yV>KwhSl^`}YqHWX=Q$_j!u}o<TZejse5jd-qg30pPnQ#&7DXCjWI})~~1h#fXo6CtEC>QLCKI<DP(qP)HV16F|hIisRgAKqk@ISo?dOo~PMmN_50tQ*Fg$o~wwn8V06ARX3PrS+WU<xsj|ZsG>k;32y7d)m?Jm9cAbKyI|HT`FF<J?Cr!Tr~Ia3q)9*rdJSWi)gl%IyaXN$J!r3i#B03>!v<M$<?HvY$TwgqWFuEu>E&0Boug8lJL;~xz&1eE=j6#gkgO^Jbro$kff<aCYRH;nHaViW0q`U)aK|xbJp!@P?b*;I_*Cw!yYHc|<w0YCWD>R)c`;kWDcDto7JyIrD_i@W{LCi@R<fpDHO6G$O)V(`s%W>$Ab5x`dKI$OgwaSUmY^G2I#!aOk;6@*Z11S)3v03v30;EumgF^^4gLKe;v5mkE5!8tl@CUgL5>8TLb4YN9}Bd^>Lk98q=_E1TZnj{mg?y^>Szq9HKb#=T)k<#8rcMfJCjX7L}jN2Q5$?AoSI}wmP~A2l`Sh;>bwCi^`M5{bg8Jdz$6gz`MKNl-ZZKe7>yZ9F~yvC80vzr?ddTNT6a#9-q*JA{DV5^QHz;v8GIre3&AJtOGheOavl>fkRXcSrpf8+7s1)_XIx`WpFKGNF@mTN245bZeeP)w44P<_1Wu`OSgnM>+i+Ov+I5(rw@k8`>)ew&hFf&oc}|z%{0ak*qg^Q`Wk#xeS%i%v?s?T~tR2;SZSQz}jKU)gf!6spZLK^#`T5CNaQxXb=wa;M;xT9yf^^J2Sj)rV<e-6(!-_!bZDOLNqE%Cr^^|S_t&_xyZyJ}W4jcZK6k6b=^I7br$=XZfmAA|;G@^Nngb|vVmmxHdQH%mA{%l!PNuZv`R`Eusz7_JRB3lr}h2L%z=@GCRh}Fo}qdd@(K_O~I5vlWr{4s+1j35zjjL&CyRWJfwR76mR@={R##98z7Z{(SL2pJpbSn@Ci)`w}9Lep1*=g$FrxWLDEm~x|D6cum$1bbgzQYxB*Wzo55329n_=RD2b;N4P9<7DmL?NskGyQKIv$fmC0-GQsZX}uBJP20f^sw?<#+s@hMyS+u)W{%ntwpz}50+!|lB7cr{&HdVM6<d&Zg_EU{by^mGC!P-Vmr?y`Xj(}wtRToi60hS@#=*!{d0f@JBr3UTaIBbX--Tg$P$YXEQ)pK39o4=yv;6`(+ZRc^T$R?fHFnUqG3X|8XC@P)<G6-t_95Q^oF2@!C^RD7>Cm#2JhLCrG=#(sR=G7ZtuwUeQML=yyAY0Q%tB?bI^WW?hLo;*>x3h&6Rr^-MuA1N+sV&O-axX&x7gHZsU^_eH)g(p{k}J~YH?AW_sv4HA%EJ3WhY0aG)J7Mld);(YWbt<rr4s`z`zmUi%u0mI_bgNvwJtJrWUg`t%<?v@IfN%$y_(Z*xs1o2CUK|so2WLj{ISd`ip>261kI<DCzrNhG91&a2Mt7#-l%JTL6srE?`rSKEMC{*o}pMHKpDqzNiB5zK1^<bep;@K1MY-r&sO|DS}7cO2@-~_zo3bfeid#R<p0bl-8M1jKTntDqKH?&s_0IUq_QDe)5peb8H976}u<eHr%1=q&H31Hka~!Lvx)-+sE79-|+SJ<XE@tw)HYGeA3Fa*s5KJ{yXTl#o{JCEPsyrB)z3M+cc)bco;^5*h=5%IT7FKX_Qoa#is{R<E=Edjs6+@l5QWY@E#2xHo{h^s8Z|{UUI0Y6?bg+*A(8m|7kA<tvD)2Q8M*!RhWBgb+2_-y;HH*C9tO5mK;hfqn>!H<>LG9v*YQKB-tg8{@PzF($2=4*o?MeEawb8-c8H!v8p<917|qct+}J_Kd$Q?amtgsfowadQWh?tTopxd4(4<;E4BHm+eCX7&LNFH&dQTdZQfbCIx<%Bc4k*WtTPTeH(R`GlPsH+g68WY8%W;e6(Dct<DA`ehaj*!0BbkV>!Bg<Z!u5W7s-is_f|98PR;S+4kq$+)8SqE@PXxq8U+sYZ+Gk0UBEUT**bp5`ENQ)vj32<)e@}#4`2Vu!dIyOfQVa<vK<m?ADdL$lF^cRERl8=-yg7(r#-peo)FUC6hfM0)m04fwFUK;1iTm61eAw!il)xT(uTROX(GMzw;u+*)=vVxgQ9l!BIvbH(O<+%BZBehqp?%U_)`ZSjt2PG3`Cs=A|jq{)whdpRdcBA{bF1|Kv*vRPnPVin4D%41c!Ad@q0}VujRlc&^JJZBwYM3IxgfcaY&5f-TK?Vh460m^&J;Wc-5rkOFrX9l+EG0&3Gg~fJTP7s$_|s9(esAlv}WT4BX@mdC+VU!@c?wQOi-n)46Lr78PERM2BQfb5lL*g%P;Z7WMu85wFtvAV2lDP5cKNcS3AFhS*%&xp6sIjWM?xov5%OWz%kz6dsGDu#D1R1&>C>9{vM&F<14|Z|OXcTGdnShaZ%8AZAVR4SkT-iJ)5T@o$dwv<%>!21RyNT?MiV+XB|r_k(BhMDtIpR>@Dt7l_-(j$QTe<HPFpH?SX7d{$-aTWNSc+bSwGUVzVQu3#_#^BDxFkU_u=W)Sf9o<JtAQEgcn!7(^kq!mn3`|l6PKem9%IQU;?F+=A',
 'crawler_snapshot.py': 'c-qZ<ZIj%_dEoc`70{}D7*MmbvrAGZtvIr1ijG;9D21YYms*Yy><nfH+L^%!!0d~c`@wNJv3>THeMwyQT~d+kOC@E;k(2mxv6wr5pi2LPRi-3=y1#JGTQ|Ca#_TLD-$i9FF+lgzZ%;q({q*|!`j?W=CF!fT?s$VNI+|>xv&o4!PbO*6@ouNy+lx3m@g~t;JjvVMG#%2vMHVHqWVY|6nU~Cl@i9Di*4Nk9Mp-)b!f>=G7FisIUNW7hS>Z*qSz1IzlFstAHT5prpGR39bDcptnZyGsp(@=-7qcSH+TJi8MT<!>Oa_H2JR2;sES?n|Zb{d<JI>;0_<EX7ZXU;jMUg`FC=UnebUulTc&O_a@o_Op_EbH2GmU1^ew=ZsVN}FLGL2Ok^&5r{f4`E>r0elCN+z8}kxX<qXkb`AEYeWbQv=`sN-|f^<0v0PH`Gu52VQihaS=hqNPn3}!yybbS1;bm)0ujnM#Y%>GKVK)Sg+#;dCsEQFrDgOx;yz8yRLtwgM+xxzi`p=A{ppr^Ar78&|JgH>lel#7qesl$U;M)o-MKoj*DhSbr0UXl_s+`KIHRBQdmW^bT2Kae5NYg!B?M8Ch_a^TI16EB+9b%2$mXVpz^YVi@zvh`Z%b51o(jB1$}(GoE&(U7v)H^Xp~GMz%&_5F&U~Z?Z-tpnhe+0zI5}>8@FD&?e#qW@Ogjj&FlE2HrJgvkE3ibzJYrz&NjQ9Er0FKYoB}VZg~CoZr;ATq^!TTwgx-ETNHy<u(RgDe_0G&nQ4UV%+jNl!bw2HLC{I^bd+X*7p*|550iZuQVSQvh9W4~$;Z+5)n~atXB;0(1yvs=AP%u?3vo<993{n=kc?k;(s?{<`PrTy;DnC2kfSXECJl_*p3iOju6Cm$ohE~D@1%%xYNhQNW55?McV!p!TU=Qr5YvNUlC}6p-n*MEVE+QW%8M{Ppr2f&h^KP^L^W|#;VhcQ1hRuq-ZS3F@0@Cs@8Gg0Lx9-|&N{_(?w4RXM_B?ppL<MwEk9-Pd=d>}=_`+ri{pNl@uhoj$?zP5$8w$FVmi-TP!7g2%aPPZ`5;MppN}Rvj1NijtmthAop^@3u;nj`(Z+LrpqiYd(XedM#s_Ku3*;?hWpUgglE@Z;gemA%fnvpp^b)pK-aH21@8G^~8Dq&@BMD&I{d+Ut16=ly(=gnoQH@i;8z3I`cv_0M(5DU-q7?;@K49D=J&H4pZ8O|7ljJLL3&$#e@DXssAZeYloPYcCH~co25s+$;{d*a7zD+~0-l}&d^epb=i@jFn@7^oCdy8&2>h_<(f}TJR%t7hW_hzS;+h+_=H^#Gj+{ue9nYRL(zGIv|gwx^4pbi9qDkI~cLoZ>gw17FQ^-C9p+F|&%1HZW|$;fNr$4$=%zL|_**LHA4VXU;laz(DF-%$c||4Y$i5#I#Do3;Gc76p)HFPb1~ogiuOB2NK&0Y<n$MnRK07^f-xh`d>R<ZWy~qaB&NJY8gi*y<m@5<)P-J^gm)b*A#jIa=m-7{Ywi$p>TL;UO~w#Esm0s^{(c$q?rgu*UCWFF=F=`p_v(=5Y=LHcUWd0AZn3g)mCu$*_gB11h|97H%HTLF|cvoiH>n8G^Jl^l(`~zJkV4G+In&xf*gZ%z03}C8Iq(rnV~DqrUe8{zr`zDt2az$poY!5Ug-S^FfE6ffNIipACu@lnFW@FwIW@ZD0~6K+o~rvLU>h<hjU#!v)~h0N9dRm|HLFep5@>b&=~@{CbpRK#LnbKJC4E=M^tn6yp?#<cXK$UcMNOl0gFWjnI0YrNB&}`4_xH*pSE)y%7xZ#M_JEJN$@?!5Aqvw+2=AB6x$l0TvD@Zx+%EAWr4%(V82B#%j0}P>Bk6xM-^jE1be)O%mi-WqFx##Fzx;2E?ct?0kUG#)S<OHo-2QA~@K`Umh35Jpb&@=H~8~H)-RSgv|V}HvJ?2){I83rW|H$$2*1EPi1E^_3A&6|3Gd>QkG2=8Q6rTE(g2YeN73?8oVyMT(R#x)AP2fgUlDxmbr+XL+l)E{=mEDZFRfd`XJ2Wb=UxtH!_e{??r-NgA5Z`mVsuL;isOT2$cX2t0<g&+PnGoBARR<QvgadNe85)uw0YHxGl0|Z$a-62e4euUq?WSkQ*^xy}bba1|}ADln#oB2k}V`+cJZ#ExixgUYA=v8YkEf7#CHuCQOgYMm||N_#aiNVH&;FQ+chrZbVU5ZBPAY!_>=@!B+2yHqp65{|MoY<Dp_wAe<kju*x_krS{gseIkTAj%RgRuxxh*)fpLp0M=I95_GE8lC9ac_wAPTv9irBzDsa0&3j}H+9)x2H7Tm7Rcew<6J)g3tXG#A$uWY%m7@yvRm0DnXbyt-uyu;`e@9Ohk+mI<$dJ!1&{upovquD4z5^pj)yJU#IYk73|L+6yF-9NM>x`sYV~<s@cd1+-;9sQQj7_|?d!FpLR>sT$XC9?SJ@p#*OOcZ1v+co#?M<TT-Z1jA9WN6O%Fv4_E0A)hs3L&m9~DJLI1l|vh9P|i1i20B7eI0qNayOi7M-XTax{94%(g`>s5i?<+8ZD=BqQK)DNv$~jbymN&~Ts2Pk_{lj|Y=Qo*c%379K{PZWlQawp2Eh#}gJtn&b;$wXh|DL$*XXda>uX86pf^y}ai0WCeX|o0ZK}SL^iBrOaGf667<*hR@2iQXO2rUdCqOrf7B>D4mO;zD&|s2Dn0E>8SzfjEFv}kRrEjY!DAR)!@%M2G>=5;grX=<8>?KoC)=rU7@p>qaf2#)DcD}bJI8Gf;$u_A$?U%m(?^8+!+Xo?Gs!zhRIy2R7+K1!h(=RF0KT1PIcdXoElg@pxnxQ1p?<Z<Qc<_Fs0=Mu@fFFSyMho$MQKkwm+9`Xb}@g12)!$s>GJz8P`cOTY4)(27KcO0}u-#D4>LPNmDv$t0}6Kp=DxVXHC;(D@E2s4Pb8p@`O==bPWwVi2g8Nd5mxT>$iSykuC0mjrDj_nT2p?fthlY>|=>Rn#~tEz9SBWugECt8zbcPf=?jD$QFi8ou`w-7?@sLFQ`~wb_^SjC`l$ToCc7iu_ah4wLyF*Ng@8>`7sJtJad`^n;lfN6QBv9qRtYvRdh4dM2x<rbL=-r8t_X?9rTJRgJ6Lma<%`?vAVW`F+0nU+C%`DhYIB<6*kNkGe!GL$^&}#VUjL#OLNub1H*0xuaF+^_ATiLjh&hX*aR(|5dBsSY+^qETVS)m^Jx7Psxq-Wqp_~Lloo*D(BVs^U<TUaBDU!$H3BV{wk3cPRH1V*?f_C{-5V6?)kjvj)mQqpf+pnOA(O$amp4&qoF+NxRvo{g4`{*zTV-?<qEQMhmcTr>yIt>E&nkh68KP-H+pUYD*JYPd1CZ=_SEJEQ^lNqYQm}mV$Rp4h#%QQU6MqY8F!yDnS>Tka#7>GJh`*W536}TW!dBDF;RtX?G|nAM#{=jw*paemz^%7+)8s_~a{^jD{C@d<l+1t*FgLHy8_%_{JOF=Q5hN;L44|U*vZG0~AEN38l|YD~n)OmZV_<P%AiVikSSqe@L94Q*B#@s4G?4iT(jHvowm>~^#7qT3)GK>P%N3GY)Eq2#SvmmpA->Whu2KhFXHy1sz-^EAFST;6eqm3PeqnCrvaG8KBy2T)s447XCQQmXFkgnx{GNXZ&7FghtT4^Ur%jBYNl1VmO514HJcCyw;5K7f!?m@+B+7G-Trs>?ZXOTfIT{3nFrw8o1O}EAVF;?wWJK~X5DWBM7T_7|fl1gBvB6wO1yBH{hw|fMj=E(>!(#|Y8o6ZIqtByRE6y7G0HeVUOrn$CRtIDx;7j0t09?J?_Aa4)9#n@s)NheJwd=$}iow2~!K8-hV*<dvt*Z!9KC+r1UoLmyOCuaGUS}ua-eL$i-+R7m02#qB(fD5!6M|8lF&Yj%(U{~}bzgB&TtXRjIbs`7W#CX#KO$$Sd=*`c#hJ=SQ`JC%*K)xRlvI?k#nssd%YNl1(A6r_1sJUM&?G)j^P~XjNEwh#nr*hmq9ZBS6<~<}j&8M`yrcZjcOwUGQRjzM;hFeU0yE#P4yi;p6H@thRY(SUmVk_xXO?gVs%hIJLxbq7l|?|i)--q6iw#>bRR^mT8GFQBT>spHMMsL2Mn#~DVKe~vCq_sQq$GXg9)WbzUCFF`3d1<lh{n)JMp78pglV!$fNHG<OmHo#8<w?L))!;;(IVw4UN334vA>8&VlBP8uR!!etI`uN3LoUs_6+16*dTUK%mVxhciCCn_`h>(U_wZ;6DnJ4F`8KYqBrQIQ`8>Y3<1}K+Gp`TnyOF8EMnjlS*pviR;Y|}C30QRqzbXswg`i((_%EAgO~v<kXkpI6YT+HNK%8!!H|MRRs0Ipmt{Iqf=s0xf6;kE+6m67191b<5DZgFU^?~R1a))c`aXzfuwmc44js5YK*mjqGOXg}xR_4bAjgm~0YGfdhocwX?sYrQw=Zp8qNl6=Sx}8&sJG3t@)Cv03_wL3r$aqY$|cpZojDvH#1`h#HEm8KmtenQEDXeBT?BQ0au)eP>6N6?rfXYXfvl8OUd=&#I7(~CKm@>$bNqzeCM34I`bN<&s%IoWc+UYtzjEbra1qcj8mNq9ZHT)Op&HFS?Olf@I`N8Bxwnu4#fKP_ri6TmQ^ObR>GNo9HbF2vL{}h4sdMm!7viQ-6dDOI(PoNvLOl<otCHC$ZB^#7Z4DQywqStu73%fWqOh#+TsLTN2dX%(PBIVWv4!2)z0%+9_AzoN+@GX-(WHq;n-jU1<<TfsaQt=^6O1h@dn!zkeK6G^gE|@Im8i(ec!?w2P@dx#L)cwl`-yuMD+X-_3&+b($Sry*I*MJNNQlE6be~x<IbjLg6Q`P88L1me$BO)FN~ei2K(}yG^-xJ{_T*u$PBeIX0r+Cjl6WGEVtrRbM;m0f9J{wclhT={P$Pv<5Fm)>B08+f9FQAgodgP^`yE~hkJ%KcjpGQIEYO3a2>2>^*9;3W_RQE0K&I^`u?ozT^ws21WW2z6BAbPy^HuCLr%>DRaJNw#FZPzr#45OYvANhgQByDg5onkm%__DC>Oww=<9Q1^Sov6;C+)^Twp!`DL&@c>QVKHa3HSeVZLku#Ei#T7Y_Py!%qRuWTGL8BbXFj%m8#<w;Xb9AcbI${<^&A!aSILkB8-+D-1N5D!KQ`aN_B@_LQ5)2wg?D=iEyYbS%E^q6mc{a`wRN-kF@D0S0QBJz)=(6p0vSuF+-wKfJuKsrdHT8&XOaFMS{}eR3xD6r54jy>)ESUFFzYJuMRiz3`f=wte~+hDwb6I#bvJ3=5<ejE22jVopK}pRg8uy#?cJ?SqSaSD=m(+;lR}BFYioc<0y=U{$06Bd!vzt)T$cvB)I6f>iFE;ZC+euuv#+V%P8Qo-v7=vq9@X~Mzn@eDr=L7z-?tz*7(T<LX2H(m!dUnoyCA+=Q}sR?Q{!D&RDN|IU&QT)E2N9L7~Dnc~0HEQPf(Pl4-J`e-KT&jVmCAF_b8;Y(L-ju68dk!49S_#z>VGKz92(s0(gfAHnX#Y3?pT5+}P@GAi{`RBzF-o8iY~Gc?re<*l9DouyrJo;{E*cJLV{c{*=tnSeeo>AW$3Ryl&UcNc74^7tAM8kmVA`Wc+xcT_>#xNHb8yC}T$9sC-Cg4JjnSQTq(u6?m)hoOO6V&ak%T;0&<>=Llru3j$L0OfQ$J3#_z29|8tnYP*w1`{?PC_8VK>?bobSOmYKW`)iT`iMc)n_Q_=92fqYB@mGw+9(oflGxLwpJ_DElw(3o4D9t35XV9O{wa6nZ0A&CT{r_H*qN`wxr1PLXY1M0^8&5_2xF({-cz4zgZ9o3Iw(jF7v4~_Rj<103%Y#e(QWnf6={T$C<6Fm2xX_gL>ka=F(d|llYCK{ZssYN5@3~M+yu~PR-Kdu5h&&ig;>*`z>uIQL9!H7X`AE>3bNX2yKTqMcw2Qv^J4r;&9gF|&;WY=9X4fAKkHJWj59odpsk*7VnMh}lr%?;Qd`|43S94C&DFi#%7XMu+x=Qo6ZXVfgPo`ImS2v+3ePgE4>Ep2eh5&<Q%|wrAP8n+?gM{w8$G`m;(#36woiZz62o@3<7)Z%nzCFX0-vwIa0KeSM2aRs6WS-(r?N?;wGyW>c!Hk@r@5aHs4=JDt2q@^d+4m83AHME+w_%`Ma{bz%3i!rp_S+AVCV9|Cb5TiiD~T6qu=UoU-2%{KkajQ)VcpDrNm+SHY_Gu$Zc#;24I>!&ymfPNY*TG-~y=aY{W10)TEh?`ko~<-QX`9$lXzF@`DwpO3Stau6KH{;~la;1iT@Lw5lO-e;}ngQ1P_z{YjUkrQ!}Y3hA_p<MtKB-iVYKh+7n%g0mHeg;*uw6g@NIB3L5!A1QyN5dTeh1;*M<#SEEzlxz><a{N=pozBAOR5s^Xn4Q9QJnQU##cyoSI$x!nr5MwI@dXrFqT@<9T4Rb>+Uo}A;Ex#%LF_-HxD<@Y0|(G?lgKEdqIQ#`{tfSTRzp_iJ=hl10eRqRx7%+EOaK;=^SejxRxALXax`;>VX#~ab8GWW;$LMNKx&&~{6P-t@(f}S`b4!CK<FF_F%dY76<<m<)o?4@Qpd^kG|jShs&uFlX0vutJQgs<D+<<;)$Y|+e-(A@au+0&%iV4Ka~c2qd;IepM^#xA<`uO$mnaG)dz2%>B@q``yCt0d67Am=sn#rNG)KNS=<~L0hswMXnuPUpR6pWxJah^~FpCm%CX?9^U1ppV#C4-+vGNEc(K$L^=r=Bwe+^w;zB&d7vNnXi>hlbAcJbO%5K~21)lh2LUQzOZAyeT_(as)f{UXMa+T?!W?yIkO=mG~6@KP16K{`#)aH;@Ba_B_m+;~e1UCZk6S@l+VW)k^$-bnZK1YiXkIzBk?t77NqUIC}sRyWxB-3?*au|?|rfDL@5(caRyz9SBhJY`U4{>8Qqz)ZwGDYEnz?%sXf)6jy8>`z1Rw+03ld$1-EieIhlwP(BC|Chrpk$Ls;vWdkU9BY<!-KIgmajG^K)`c3BvsJmh(u(1u3V*W$Uf@I9n6f}Z$eYw>chn34X=yoHOsl$gV5xe8p$F%qP^%JhL$EbyEs{Wx!W*wJ(DNBFGQ(CQ%xZUo6(wRM-DZ@s^2)n(wB3!Z{<##ag7bB8%S}y|ttPqc%e6;;dEFY`KziyspuV4+#)RdJsljNmb`AtB4VMzC%h6wU*D&YwdQL0O&I&uN;!xM}&@!jG6tb6lFB%-ABRm+t?OnR$wN&3Wz>);!Bstg#$qq1q2zr$}+gF1or?UY2EnUw~<Y6|k8y6Dnqp&`)Eba}l5O1pCvJ93RUdpU);PNkIkFs}_>IV+Sp)26)nqJW8#I7m|8k>6h3DhE6G3hlbC1Se8=9{&JK?HWHaGVt4IITuLY5S&hO)P*$gb7I`T&#>RF~go3Bc8-ijqLSwE-^J{a26hZ%;Eynd4tiaD2XLTq|%veM~~P6do|O!vb=|a;EtJ;#@A{SH!RKVcFd>Sfq2ak2}s*UnYTHJb2bWGw1r!2@LMp<T?W)I!)Qv+fed`jEi=|u*z36fyEriy=LjupbCSip;WSExu&D`Xplfp-+vHY}C>m5dUg{B`zYV2o4&pRs+u;W?15e$QM0Jq&Ae46*8?f`2FCq{-Pz)tKHN$Qh&p5qd7?XtI;vtb%lNAfxOp^blM4@2q(?Kx97et8!N(d#{;$6`$w{V3p>i0N}jS;0x#f<jU<60Hb&k6n#u|-pIGJYt5JZ34h8aXRk+aQ{SWDaZ@DZ*SCL7MfdB&To~(yc5~%TCCiZVKy$^wMmM_FC6Cflhl=>L}|vzO^cu!=Wlj3h+TysYp7iFSuXDM@pq--N=Y(RIafksiRj&vm%~6Bgzp^tC@i&mVv2!JLbd>mBW@{>2Z0kvH(sOg&K;A1uv0XaYv_MCc0F)n$0qgk%qtuvbK>I&sd!~$b2pO*x4P0>K6Wr8#X{m7J0Q91j`JY-zcu#V93JN*i|&8@7iR|=G%H@eq5`%${KyBOPr?{_ns|tl&P++>LF`%cs-G`OI%&b<HcwhtSzy0n@VKAemt!j4YGosz0~@z?CTD#>T*@9GBC2LOf_p!&14er6Jp>GLPd{leGY;bDY57AV6DmXCsI49P||^>fT#u&bk+RERw|5n>f)bmihjaBS6F*i!@jo4sGs&;NkK+tZ<h|20bvwk0F|PILG=-&b_LHEo%dv?^<vgpMNE5g{nH%(1w=6jGJN)8QLh*RMfP`4k+gD10e3nuOlyUd@Rs#Z(FB9v;nLTYN}?3YJhl$kP#Nh;eCt#5>i+hgT!%{*xYa(nW~02=fHcC%i0!_muQoWV={ldfe@P$veloWb&2yPt35kToABq%yavSJsW<t6n!*FooQVQWpt7cjt;l~PRbH$oUR0nMVU#Edhr;^5V0xWx`Qd~mr<}U(Lt3x*AX~LP^qoWvmC_hJ85_9e|R0$xA7;Fm3U=nXYVzFriq+=W}GD;RqnV<!e;51n&P!Tjb9p}?AreShOD3H*ZrY-urO@A-b-z)U@D*b&H{}$rIA1LoWLBwP%v42*Ie}`a+EWmy{G)2Cry~*(uvnft-+&P?5JH+#|AUI9POa}=1g7+FG7mRV<hY7Y385NLrqL41*=46^{&FTKI{SH9ooPVyfcjdW`XkXZ$sua``jbU6D6r<gh4<Ko0L32@@OlX58qYRW{e43@k=R?|cXv*`KccBC?s|4Gfx&_9xz09xH#f4Qnq0D8fCjtin)3dCWE)=ZU@%0HNSDq0NyFM=~6{wL4dfUO%@{Y48R=Z_M_uLjlmM|m8(lLAYJwIriz4NDQtsIfsie`dWBf+dhT{94!>=x{rW3Xg~^TnPz!bs7+EjSVpm%I87`bsfoP0<>`UO(sL2}1bVkXjWb1%!~Z^aJ6in4<`D?%>YXn_2+iHpZqp$8!pG+1$Le`%*IOtt-k8Q6j~9zsx&^2Od;iN?5X0OAn0}i8VVWO4Wd~ajK<`ZmIyEY`j6^sJ5hHq|RwbT<X3jldRYPw1&qzh*FB0&Qn8)dVv`afowq9&fEJB-#PI<_}&L!o9*ArFYUnpTla>iTkY+$P2=AyXF=<;Pu&{^RxV%ZDgi#Xqo!(5&HoLAV@_e*1Zo#UX8g|#H$OAo{LCLf5?VqFJ16)#QR!yuHoD1rgvfpBJmgM11u@d2WA53M&M*)XVLm$G5Jw4$R=Fg=Gy*0e@iW<I@R`V}LN-a)KR?k)_<>8|Zxa>i5EZ|+_tJ>Y#}4TP66K(i3beMvI3K`vjAoeRsD6lfMM7TV&=jE*v)W&@xGu>h^nuUpVv=aqyaUsw!TSxc5x33;=l%#o$Y2t0V8XL~n8FR&T@;`pP1Px{kZrZqiIe-Z1ZKU8;hU~_fYu3w-fFxdHpf?Xv1Jf{Fiq|Zy3<<s*4rhS+Q{da-1;Sg4F*8?N`DC~JohUCHMn^W8e5B=urZ<s*x2q%aCUuEZL^$VOsmkME{JY5J85YeKwUPszsivf@;T*ap~<~_>$dmGt%u*b?fw3n*Khk52wp_2<Y_^)q-nP;XWUJDd1#)78RSDlq6>Xc^2`BXeohpgQpL1P=>(cWBOvW6D8Sn<=$L#)imN35>3~x0@X<@P*@>l?=10X>%6%>+H-G=%UH4?|$WCGGap*yknF8jDn_Lu6`;;&$2l$>i_SSYc$`3Hi5jpqK(NV{EW9(NlQqT<}4Xg_?5aysNt@r(!Wmea<mt`5qC&S@ANPvHURV$k&n&n5#5<j(eW;s?39gPWR(xa<#Cl`k@rvoXnQZzIWJl<Xz4wb+$?dswrwOf>1M&56%E2y3^&g&Xd07Wx+=r^tC`J*DTvIJ>h!M?C%A&o#e<yf#CQ<|eR{Y+?fRTtpMz1d|5(5S2DKJ|k0-ttUww{TX~dhr{gPg@c{<2xWp@-{GhK{5>9D&gA^Zu;<$lrYK5rMee_-kYh^S?Z;$i&9U<qR!2b;AaePNlgk#RjpydY8n*^OpWUsG!GEF&gsBX0=VlxEjt`M3mv`DaKs&ZJ3w;8I++>F4#13Tk{RQTL1thONi({D$Z;3?XQ9h-1RHPBcdLe8!nUVJh>`>MNK0vYNnz;HuwUB<(qH6*_Ei~l`w5?XLA4*c1a75xv#J=epi=KXX!lP@n<VX7U20GgGq>oXN=<v%S{SKs=@h?hZwHR<>UDKBINtiL+KZZ}N09-X9hD-+K{ZyNIE`%viXkc}{Hpp=H9qd5iKipW`Jz|;7>N0&{5IZ_=g7uT91t(us!YhC6)*sc0X>m!aB=;h(TexPdV<OyuEcvx#haz3VDHJ$7#`k3LK5p+2lIviY&^PeX%2JN8EmF(l7g21@qhc5$6x=akG}QSzxnE){-1ySuehG_>id84@lW5uS3%&4v#d(8YnAYJ-KdPOCW^TXOLZ)$vS9qD{k!6#203=JGR?+TXIA>TnOi1PxU7yCn{ri_tDa{AN@5&M3hO=oZaC6c*HW|uW^4^WsOi2R2!pH}3cwyU#WT3gs>7RfmCx(1-g<eXR3L87_F)%}y=X6|>1b<9^XrSfSE0ZxI!yLCQ&)!09P}2KMk%=m3~e)T`HQ+#WwW)rwedXURf0d4g3ZtRZ3l<b3I%7<D#i_F5M^9eS)f~#V5uz3Hw&O~yOj1M`JR;u#bA;n;F1V#_a%hYTR+8r&er>zFDW(OSSp6!0lGVYL|I@KtcjcGNo&y>aoIqlwEVVEnnyIB(lKp4D1VUfMU5eTDQ#+0?kTw-Z6Ay^^Oos5ubHbz6L-Sp9caxuz1pdZL#Io$0$bL2VrDw#BpVyrbYM<7cf>j6YJlaO3)xsoMwSe^I)iE>ED<F4H&mgTomEvn&#b5lT69-c*8Ea>tUfe@8>WeS(cy!y?&ixxYs2y5K?tM-aAOf-*jag771Xd2<`p`L%gBHEwOc!g9Lbpp6k_rV{J!g7myW`ISAFhV*7Z}(2c#Fv6*U51RmyBNN|8Nj&QPYvK%eWp1g^;ea}P88dKzAzFTZkE<ZpRnUFj6<<Q>|=ZT%8gWQP_^)wRlp;0<6O?GDA(S0FbrIW>MGf4-g)ce%CIG&drXL?tZ()yElje4}Nhk58NX1ob**Q+5J#({pZeE5JiKZD&Y6d;Vu$fBEwpBA`_gNU{`W0IMII)Ax@~!P`HwLzd+wU!y6xt+<1jF`V4`KDI((HSl0dL35M_aA%cBjIZdbO4`_X-3mu&Fe$~abOX+C)OX2K<u1Wa>Dv!Tju0x&QtJ8pXFtw^hM0rOKH@2$jRUnTBLaiLU;+raI0@$>{U|%ZoKy4j&}dv#UTg9`pMR9YF|Ji&m$y17DFtx>c6YX|!;(@?9hdaZtFW`SG>0SDmFXs&&^p;`qoumA>~)fU$es&lKqnCf^3cwT;nGBXUoZHrL7QSfIi~Z+Tv8x(!9Nr4_dd6Md25?jAMohLNtVEUy8|{03n;melJ21eK}z0&cdG~g-@Uhf&+mV>wR`VqqyJ3cSN9Lpyk#*CslH<dT0TK5Jh5BPUViRMUmt*_>q5x^z7NP3`+?cMx~mi#65v{PDZ^Dpx;w<BH8aEDM^&E;K>K?*aoE=W-U0q|ynzqK>Kg4qnXTu#m%GgK5kWvvn=jM>UJ)m83=or?NiM+hOrO#CPILOl8dx<gsxCzcWDqfJ*^W4T?@hkX^ku3cnVeop(}Ts_mB*>pSAF6tF>YRE?UC3>XExYNI<v{NGfO~ZOb?A10<!p%`$FA=4FhQ&rnn_A0@>Csuo_*r$Jb@H%J~Q?Vb87%R#ZDC;ffE{y2f3$ZK;=uc9uAc>+39Ji-!8v+;U~?8SU9W0~bLS;C5_{aXEEIakX+DXc~GH1D(=^t0DV<d%oPlK$jgW+TN?Pdgp(1U(XP!Gsni17Z3A$x;0GQivb7U)tyb|(IUP`#JFu(0vy-{MuVgMDA}>O_Y^y%xKhb%0G7p;0vAUr3*ndL;+!$54jE2EV#%#-NNM_$ys5TQLkP4Ul$+H?0)%J1ID66!2bqj!aDj2DZ)4aZx*6$o;$DuCAypNebH9^?7!yUt6!S$%v`dFy8UMuVqb@27`QCOoc{7>d2|Nn#byJM<U3hJ8W!w95!cy+#21n=W^rUJ33&&Grz(h({LMJM<D<%4T9}P{Tys>X~qHl_{NM$pfarGe36R!yceR0GGB2wpYb^+oYle@NgCTl2T?Jn%p^4O-jqh%L-=_vKFOPvTho2oFtkO+#7M}??J*y0qNM8Y}*j4U&|8m4HpvR<??U^)<)bHRpen|H5f^Dgb%&Y&EJtsDQ7J58Idt9p<qycoi}5iuGV1Lygu_&eBw%Xb@&)V(JF5o+N}Q;(t#a>}0zJtNIh=EO=mC>!;$onGat_I7E6?J3-4EqvQw#?fk;Y!7E2I{+Y+p%wdQeC56`gX+>_d+ZIVB8NCaTnO4<q2p`>?r9USLNnU{tlqNu*H}iEY4_72DI&FS6%}FPx}IytqCnOyaAd%;UYnPRk>^62bX_IL#LGMeXL?iad8^eiOY7?vq=seZ<l=G%S%||i&h?a0*LaateB@UGd+f^PNgE8-63$wSv~8+`2@I$WnOXB%v>hGX*w&Uxc&~EPw7rVJL`MSF)b(k38L^EdLpv{6L41o+^MlUq-&<V1((U?~P-TFB;KlQ!5g0)zj#s4+QieB5Pc)TqcMSg_)gGZhogw8%!*hTT<}XK+!QxSH#0y;k-oYb{^Km41_m=SM6){f_jv-f(TT~-gHLPqt`leV_ma=7T%kd-VHr&*Sz_#E_3YClT8tWNZdUeD2LA^wH)Bz%GT~XWLppa~DjCWCZaKq%u3!uU$xk@!nap=ijM)7GA5NsBxofE4-D_qu5I|QfS7%w<ZJKH73=$fz?Um1tk8tb2wpQF^Z{oN0~_V7m^eDC2;9{%FPcR&332Y>YNk3RVB!*4zOvk%|-@V&l@Qgs79Of0}Ag)obObO{6kGHzJ8TOeP2GKK3iiJ0QFZg*+MbVZ}*JcE}dMGyKsHQJWKx(JX<?XUPH=KOjfw&43f{F?&KY525xn^oRz+N6_9fIQE2tz!q*;p)FMqg-xy7ahpl>M9c@Xaf>Zj;t3X3LEYSfh$VXE%eIS1+JpmqH^k>Hl4;~x#!dxhdGT{sum1%?i5sFgwzylch8?ZGfGI++j15)%*?9bmFgUqlZ%t>VIZRHEjXMAm%-oxH9GNzqKhtEQ)8A)=nTw@)@MT`Myjcq0QDebEpv&-$F2lMI*2IPue!%fx(R!|+udpcAev&n$nj?C5<;o0+PWn%Pod*oO~=PV$h%iGAs;&_E%v6dFG{1u?azl%Y=#K*5F{uH_7*i1P7oa+u}}Bt%p#otK^td2O%GzZ=eEt4IO9zz;v0q|tuZA7F~drk&Imln<K%zz&)<Cf({Dfi<*y!p?^}=m{>Q)m*LRREJpTGW|N6oAKl;|6fAZcpKmP6?fBa|vKySbO;Nw60tH*!ww;#X%tH)pe&L_Y4<>UYIx1YTCBmDN!5B~E<KlrOpzVqJWKm97jJN&Qjef-lOK7Q|CKl#Q_>N2MP>tFu-@pr!e_=kV{`0sx}p#S@K9)IJXKmNxbeDaH5JpSXaKmPk);@c0ti)RPC+4#}>Kbylzf=3U&=IuTFyBXYMw18K1%IU`V;m_bj{_tH`jJ-$i|2Xxg@Xc6cTt^<VS}a<-@|Da7?VZa_%eI$-n<1kGspFAi$Je~hm8(q<@CFVph}i*>)Iwz9q&CB{YjYVO#Kb@yC~O-3krFfk0uMqs!-Jh+I>>C)vv!1Ah;0PhRKDKNPnSI{VK>)++bpL;+V{1*n@;ji-Qe)CCB(aNz5tNyRH?AaR9U(kO;JUg3s&M0aH;0i^Jd0Sn{7$G3S1&>s-|HJCY2$oAL7AyPy^i9XL>wdO$MMur=#K)UZ)O+(r+HmC*Z}%?oab{gU<HRQ^Akdp1QXSZ6%}o*XYHI_pbrMyeQ(DI6j)joYK)oq{v7M7IWCAk-}Y0jW);A%o$z4=YP5N*&Rqtc^@)D-bd90uAJpujrZR|t6=0!Q{+kavC53ROfu-<eK>7?fOZ*mb!0q4YcRaWH0VgIPhe8v3^ZJz4g>woyZ_@ifBB<d|I2qC|Lg}qcIEd+fAN!#zWKi&f8*!(3(Vc%9gH9T0@=l*2Y(9W?a}-1Cim5=hkrK(PVwmdzdrG(Y~a^t+8UpzL`vS5Y2zR`KpS-2apDqVp-3n^!JgAG(D4E;Gj13V57&DuI*ge05ipsQ`@i|gw?BFBs~^Af)%$~J4m&=w7q4;kcm(i2s4@XrW?#t&`Yz#8gVaVy(&Z$NqUeg1I6AUIIbh>fr0P3sb`A>GJKy3fmhF&=ii*0UvsP5J<sF8MR$bVx+QU1mSjv_7%5v+DRuRfvw}^nkW_C<L36B(;JZMMpu1X6QyD<SFqY@8-N|3yxl6RAcAxV!U`)&}Zr~=NW1AZb%&kEVM`9g|}TraY+tk0tx<I*K6<4bZj`__u;T@-pweMAQo-`EwfG@riZN0W>p(xnb($blY+>7#z-H|Bq)+u(cFRrA(&3A>YI=aS`IRRkY+AM-_NK$sv3GMO8PK+M$TI%_#WT{U%i<)U<`ElI9bt43mL^<d;M=daX(o>*N^>S>uIL$k8Q*lAK637hLqeyU<78*#!iT4wK<9i^h`RPuj&F1uI51fg-L;ge}UGB4n~qtZ;U_LPEDp3$j@%`vb5iXuQ!%VWKnt@V`JMV-7sJ!}HPe#athH6vin#%7wjWFi#JsijJ)<(aX^Cdo9)1f-IIOJ|z5PQ*56vvop+2oA`uMieo$@_Y3lLsC6=>KTNt!HY9ra%%@`b;x+zvN%L~LK0R^_NX(^Ghid1n@Z(IO;Ikl(fTQo0EiFkpHcF*b(;|AJA(0*N#wwPu%8P}o22_8UwKldrNYmY-9tY4n69I%g&>+1{jqvWdI`|F8ohokdRg%N3(@)3LLybzxsJYu5LkIK!W-Cne7}-8SL&N2wo~25a4e!OsvIk*R0?j(MVqy%xhvmvru}8(d4r02L@j|i0w*4-?n`2~bgqsY;{Dt^mHDQ7o8=PcpuCG*-XK#yU8~-0Zd?OzUs+VY*nR72r`vA2sQq#mKGvXD?qGkuYeKNERYw44Dy0mwH{n(o8N*L4x?P3VPqM7hv%a(5L#}F0lq}g0h17yB+xF>J#1!w6`qIriZ`^wAHu6_CQfOvhE^rvqa~P0!9-74LUuFCC;Cy2~D}D3vHs&xYFQr~{-#4#|SlRzBT~fT;x=h{yWPSoV*DNf@nV;kx`KA1goFo8Meaa2OWz<g>Oi*+s-tKh1Bc5NdSeIde(g23x=?J`5Dac@JClR3e5hrr7<0~ZUFE~3h!}(=%u~fyw<&CJL(vh3T#}M+COH^LDgDJKjccNPSWgCw;37kCVM%3PW!CTN-g;{)riLpFpSeVHGSfLzk#ykd)^I(Q2fz)0#XfDe#stal>ACY^)h<%|eg7{=OT|fy_=cFth-gM-SRS`u>3bouC?7k3TvGnI)3(!2}#DiX#5qY4?&{f&xyf0@IYgYCZRa>t{<(l{Ckm4@N<teMw(>`HgZVT5>H%tag{zgKM{z$)??jW|7xg4b?31E1o^CBJJ-4Qkf=-W^NZ4;3dk{%h?r8LaJxn07WgTxGMcBbxxm1R}RA<Tgw7U%B1L;t`Hnt&2x@UB5%i`38zj3^V>a=}V=WQ5skbdjgs9kU>#kfeKRd$zt8=j9vaM7%-f+6=`P*W;QFe7-wIynY?BGTc0l2MgF@Eo>80^kD)r>V*8@*k>qdXvzz^D+{q4KXh_1sbQiZrHo-J&Z;(M?gSaEo%9n@yu`<Cflb-;XNze}DNiG$z3f1M(+J!+jH{YU5~245K!+pRDi@u}RPGWID{;4vw*zlgvJBWWjTIOzD>^qvl_-bX5=d-ZP;x9lszGF3odE9k9mfx?KI2X_WOQ}tRP?42xT)=Midl}??lv7uq<yBVvr9!s&WLZ97gB<2lm#Q?@aM++te4n5&Yg?;wx$$BEu!bc#dMxSayUwEi3&RqC#@FvbPz)mG~*Uat%t90G_SM<Sc}l+TOjzgW4>1cT{Ge|qS}SPDa%j8)V6V6R|3d0X#nCE3Jc2FI0KalBI`lAqzG||JjEM(7x|b^K~!;?w!HTlQ=fsG);&cG@E4sk5L&tt8fZchqPrg=i4IkM!5u|nd`zD^TukV!joH8ja6uh}V9M-Q2?l(uQR^pJKI`|HV>ZEf267>4BjhCM+y<4GPu}5GiCXW%BxZcZ@wfm*2?WcGR7()DC_8_vG?>8&&)43doZgrv>JSvW91V-0CDA(+&N+8Mf>$}vjkyq=3M>SeI-Y}=oOpEkABt&Y#)fC`;?f#akB^daNZbLWA5M);HwgG8ry_}y&gb~iGkbgz2ZY6;>Da|UCVB<w?I24S`Bo|xBKb~&A{;5IUQR8aX?6{sA+Pf|K4^6<E0rm|lPd?3{#@R!#1cuk<}^PXWX1VuthpRb07Y<0pD@*HihNwYzw*WK_UB%?@kL~Su6)*#3b-*7FpETYKqdfHQiP)GVrCk!Sng5oT8fte40!>JW5gKv^7+VPlL!TkPf-T1XYZ|xP%;;z!K_&2=2a{P%YmrL^U#)eue^5Si{Z`x2-yIQ-?u;aZ+0eX!#hdh$#D5?)D38N5!AaFKD~L{iBHap4$fJcH*c$iiC6-USXHf89A%>jk0w!uYE;$`P$iX9Kt46&5>P3e5m#~XMM?>?66=T+mGYQn+~ZG8nf(7{_c}7-5mj(j$hWMb<XWcsL(>u6zCzIKduN;~EmmmMY3B`UTxPA!ytW^*QgbcglXzmNt^8$Wz4Z=zjc&cX30F-o(^#{rw(5d@6oOR{BV#gCZEbt<Q!SYDIsjE$Vr0$r*l9}hE$$<7=wJ|%xO9}q$JU(*C)pze=|f1xx4U&-4sc`MM(a1yEL+S;wFbkLEX~=vTPH9F(}{+|mJGHw%dCiy5rtzBcz~ITudVq9Wf-PIaiw*~giX9Fn${e*hqeRoCw)c-c1^~5uCn(SC{oiLtsvz7ni0j`49axUSQ;SLr?Zxv|A-9Gbae|R+bQ=f!PE6D87ecCK6|zp@NRWVbsV-Vw9LmOi8#~F?rFuRJ4T~!I)Gc3Nq(k7xGnf0A@XFoqp)Fa(9CyE=KzKnP~~m)&m9hQNFRa=Rf}}x0k<jealq9<k!}>!CWQU^LKcbsDw+)77j!C5$a1^vP*5)ogk`<}6q1$Kb<_(-6+j)ZG(p^5tV>@6(Vnsp%>iG}R5=1!LoU2`65@#f+p5X2S&(ruoi8TrghW=r(<?j^+^tZLBMvZXUzXF1@6Vf2om*g3=jYO5gg$uPh%i2@t=(!38AO>)Ucm3@k@)3M@Cd&CK^f2|Y}~-G39nQ#MS;xnCAy=rOZn`4^DZ`~VJXD&>;eN^g~F@~+fZSNd?@Ro*=rM&x<=qAPWH!4y0A~jyXJKE;ZhWMQ(_^Y9sJhQ+Al>Xg=~NWRteM*^@<~w2+cz~QZdfUtqmPD;#~R;G7*?|RdP`o>q}=22%!SkN<+i=UPXmi<>kZzm#KxPl5J@F_*fQA+v?7-)Y)6Z0oOz?Y`z?3+rkt|1EPpb9gGAlosUsc@Z5gssHIgvO3`zHay7NfS3*$)<bYH)wOEiw6@rAnM)a%En&xW53aP4hmtom)SMAcA9Jo-|i2xMgs@Wa$HnrkH$m`V<F3`1NVPywsM1fsr?uZ%`we3R-96-_ZKzKK;gEHx<4MVx0!DO?X76fIYh<XQzcT*fHoT?ut(M$s2C@lxLQs`YvsDzRvh+?b;bpHHRo)fIIOV>mW;$+h5Jbx948XZIu*tq|<`YW~oY8&8CsCS*BC@++@G!Y5V(TT=3bbBT_-H|~-x;!^fRi4fo!cP>gvJ%~qGz3EZ)bW5jF@PT~1__2za;M;6M+`Yc;P4cLlpeFf2<f=#aG*B_OPMF(1i3m!gEtyc`y8(+xwcjMYl5j2Lu>pZcC1;~`O*j5*W7?crdKtEqxJTUav3?;YOXrV`NSf?xdKcz7d-$=)HMZ9AjBwVt_I^!QLyWH?0%dkc-xpR>bfik$FAiYxI>5u%z*@^aC|^Qw{4&zjA0>44=f`dYsPHro*+}}=k_XAKKQ1*Tp;Mu$xMt`>v5!u;-+Ba$ir?}q~W&u0*2%nSwHL#jdBX^Yp^zl;3V9OahpnXz)%1({=}L51eAh^Lo=ZqOTb*LErY2XXY`-;zWC_<Ut#KMI0M3VDY?1oOUdVw^wnE;=p-i?z}7ds8{<b0zB2=@*TBp(59|joz?OOV4uw(wbs^sgjn7{21`q!cXY$d5zl0CUJ*lM&Fq5oTrT|-_E9^m%Fso?kq7P7u?BUPg<jV2FXt87=Sm4ovf1tB3Kb&Fk1r(d}i5S&?P5<3Jxbf)yznOW*fCv+e4L3U?6b-vE1NezITtFLpj~@Idu;SDU3*&DX>D@;U{wFl?;7{yM0s%q044kL@F-;y#y9V08*!nx2?a#QIxjlaPPg9(=U&?(LY8Hff!dMO;z5i2COOr<r-s6Sz28a{4UiS7MJ^0?B2G%F2KQ=OWOdh^dUfBdIKmPFN5m=VtLQagP|CM(Du)l`Az_*Vc{0-m+#M^hgFLK=d918t7fiZJ2a^`?K<2#1)I)MK0U8?i&-FZ1f6;mN&7$Jj9G8}UH=|aW+1A4$}fye@1Fz*7O%mCt^p{ocog^(OXYEfs`+x4Wa{(bL1#Cw#_`#!GZp94@_w)@_z)XshH3+2ARA|}?t@9S#j?&B`I?`>>sc=UH?`9FTCq?%HMacUMfxXy04e&0JiE(!IS$G=VOMZGI=Nxdoc1O3V{%BWYR-UDye4JuQnP^8p(Ps1F*q|@aM58q9MxcGnghkqzs+i-Pibhky~ew*B!v!b^ha8<wU%%|&7dpCgc@ZKuC$%j9G^x(&{{neVSoXB@Kt1LqGxN{$F@p$Fd-9HHL+<g7DJ9j%%;!8RS1|ROO+4UMk96VSwLLu0;@WXYiK1B9W5J@H8Uu^rTyekmV?KG%Ma^w+}FuiJRJ2tOwC~C9@xokczJB?GigQz~`+^G7;Yc~tZdRjLNlEp~LmsP($`JQb@w06?4$Jg}JfW!Aj7Sq+l6xAm|i|i-kAT{8s$gVM$EXG1n)KGcxE?;hru>o@A^w!JTi_CA@HD{14F8G+BqYfVtj1it4?gp15K)JEi88aD(q{q?`ipoyx^gl`0SBY;fi+)<J9tM~<dwg&%rUx(*%)3h2#me{s$^Z;81#%6Z*`;=ahs-#^G+q?Rq=OmZ@q0NogxD5MfU-JUQ>O$-?hP&@9UI`<7}%9g4)Fr?c8PLU<i36zT2aP5M~rb3L8;Oa^;<8Pu}*R^vMUPFm;lPXUaQph?Cai@0iA9VN6VBL)^|#0mR2%ahVL;qGHx`Y(8}&d6%0-(OF5g1s72-o#IqN4_9y9H%fBScUrXL%yYF-Ya1|KIV`WRc97G{k0=aO^WHGKIKMtQAOlS_YLlC~>$_TUEXD6OcHBi-u(q~q^<ARQ1RD5md(Xu>iYi=JMoqsZiy4>^T2xk?CoAejktzL?u?qC)7-}3Qr=??Eb2U$SpMuL}LA2?#VsvDk2J438ZfOW>JZUb#=1L)4`*LHtuhq&q)ft&}rnv1y)cexS`Eje}NaP9t^PuSEn%!?;*ayi4FAfeNF9dadjYPf)X$?>;lOgQ%_Ts5$}S8B!eGkP}1qh1r&EAv_}JAOvNbj*D1Lubo^2vz(b-NgVxnqqLAm4U7dm3r5+eUfiExoZnAbwU0M@?I|_Cv2!ZFMH^|qf+Sihujx?{42*wkSi7QoB|g5ZGFn!pf+=IG3`-Jt_JxYU<^^F&S=l+TO{3S`5PNd9({c@tB0vXqfNS!Fccvi0KnOqq7eojyK;A^-W;UK!-M9SK|H|&(yCcJf!J6nrW?BJm6WxzUM0bsK0*7js*U5xyyxG-)UY|8%YzK0c~Oeb{!(D++yM&?XbLP1#iP3!`KIbdh*=Dt#0?POd<E$azR`gnu#A-I0PYYWw#!)nHiQg9{?s8x;>kt2s0r$pAP^~(>B-?pz--VH3}{9XGT(g~iz^2s-0R!8RD70A>OlBhWY+35B85KP=4$P7<HVCVxx6wPMtNCY=64t(4Z)9)3d?BtoHpdt;)Xhy*f7GY%|jGGC@!4?1y3xV3k9Tu4LSelhVuORP{gybfd?fo-W51XvjIAyG3(7}Yeq>n<1SO^CqdqNiwtXNTg+n4A*QmgSK2Zc?I3qRw4sd7HVM0+e-L)w(#lxn40O3aw<s+Q%0QJOYBN%SuA_jla~OzqHwtznX8Bth+f)_w!Dw|X`UQ^~1DOf-30|mlsHhkjC2+(#))mI0POXzj6NZ%kC=`(fBY*Q^i26FRZ#+2(NLp9s)>Qu4R<QPe0auGLE&',
 'diagnostics_snapshot.py': 'c-qZe>u==Nb^orv;zmZn;d(@^w%NoOJ%G*b8rjB=^?F^SQ3aYqa%O08$O#{k#_9kq{2}Ov0$sF05d=oPF$~v9gC=#0qN@-6ASeHa`j_-~&b^nsBxgplV;B{%EAl?hJ@=e*&+}57o16Eu&?`dWiG1qCabKjPQ5+?q2qJHsq<Inf;>V93i&2(NG04xn(^zC>(ic(U$7K*DV=?oxBJ#L7cQ!XSH=^k*%?jaVV|3-=22b|WI1c?HN|Riu0r%4~DZ;ETf^g)OaS=p*p(p$KiJTVzy&{^1dXj$c3;O?qGzs;TNimJ}bCgz3{40)z&NM7M%=UE8`+1tkHD+Eh!3e#?W4!7?v)C&}X*SjG`8?N;w7!1+AexP$INaFSz+MDs8n^_$N09sC)XNf3Dz^t40zXAImk;!lg+-YqjEobfUXb$~hd~tKS<wp<KP5WvTV*lYdd0GBo*SO|;j9q7H{X7kWhsdG=U!Zf{A>#^r~Y~ojez6LW?>TaM%Fp6l6cc_F%ah%ys+#xPUnmEe&XdDibf(y3nrrzo<(_)_w3FN50AJdOHXsLuis9iVj}0IK&-o0+DTqu{b>yk><~#g9g^g*tVW3l(GX@3jh-Cc7dr+93P&33pg)@RTAdxPm_LNECY-@0f@a&D8Hz78H#*EPf7UbRTyC?)Aw^F@OYsMX#U0LzFz-=gUyPzK4m5KlK|t~-b>7D2lkuZBdr5do`0rZ^=4lq;Bw`RiIS+6S-{ppinhcSU%J(JkKK4{(UheUc-nSxXi!5wFbNJT;vI9V}gIEU?iDQJRA)v+#^z_b;!}&lSilCg%^4<y26N@H!nT4*G`%$$2su$;>O`MJLD9H;i@k2cj?8ysg7Hq|~MTmiddzNyc5-Y6Qm*W(Q4)B>mnrRVqQmfqmqQMJECTAQLJh^8(EXiR{%JEhnjX@g8ET|3ez|G6))XV0*v;>+-dfGWu;YpYjbX1Jfhrwc7SaCYeEqaKOlhpTEsX9EuGQcSALp$?<Bf1;(SKzTP+9ojXS=@v(x4$neo@WU!3F>|v(To>UW7ry4N2Usp%XQBFxi$3iD7OZ}DiiO-i(+pWhd|cCq>!3_G`ai_*b@Kp3xVB<VsdryQ6$EZmk9sr;xmt+hr={I7SpSXUqrj&SWd{xxyU?GTwOf##h1^nE`CnWzZN+f#A%uxL#nJrO$ty#;&`~VqcE>%sQV5Ly1G%Ib~ww@aTeywXt{<%de#c|v@}#j>LvnA?z&0h19kB=_=;IAuHIlN0Hf7vw&tOq%H^q@NO@?5u{X=Zz*T*UYIO3J&Fewr#a3m_9kRnwo|38cik@PFS6D~3?G$O@#Tu^CBrwZiPTbT4CL{>aicNgx&n4^qbUKT}A`CD=&Xjwv+TCIj<!-%iFzU8CK>j2Q%D6MNYS&ts+^BtHrR{3Bz^ZO0+Ut%3aa*GK2Dvfjd8sH?9~`;3%ZJXu)IEjY<`NC*I5n9$FgV$-K{9o*XOMRw3{qqMC`zIn8$$;UTihe|+GuYxcL4b*VXxGCU;L?UEKrlxE8N%v)|xArSakwyO~gS>Xai80#kmWu<3eSYtSIENOzd0Yd$(y}mUU{VNNMg>BFI1ht%9%v{?Mdr0l;bO?0V{*^{{j;8e8?Q><aKs4Val;jgE5Gm9)^(CACg`YAUr!m^QQ`J^#*E$}#@KZ?7&sDU^RczIyRlB9h6~i%*MPsR<|P<;R#qEh4}~Xw2;LSBccM(bbFpN#r_?c=+_5@B#a8MSS_OAh^Hq75rrESQ5?JF@%5Y!yae;BsvLuFqUwKVb=OWH7R3;u;4NHA7&>YMuDU5>$y|tCUH7}^DN$tl7PI!VHSm>cn+jtXhA!{J|%@32t_!;(DvFZXvKncu$PR(-YbS<*Kz17>>RarPb1kFCePqT%b|N5W>8Q$PSR?x4(x_*08&{GeUR@{_*gU=N0WGLGv9%1?=Ix-4*o&-jBP;me!uG{W?qyx{T9}kc^rnb9u7wDwsZR~PQx8>>y`!u_2s-`YnNkKMsZLvi9L=cd(0*%F$#ca7*0MvHpoPZd%srPc<t#(%qa_Bd4S5w)t>&;OQMlf*>!lNtflT_BiVIfJUUuS9U<AZ!nTHIGX$KhxX)qjVAWyD2cl&!C5VO6zJ8jKHp{}Fl)y)#-`889S>&Dbw3Jk6Pa~pcTDCKe(_zoLCAVTX!XO&b*nTUR+A!pZ1Vzqh3Ir4pPJ-dbl8WJR%9@i2S5j6HfOcz)Aw6h|@9b9t4rQ;T5KK7@IBKqYO$7Pq!S*{(UK7a}dM1L4_TtxN<)Rgrzey(I<nrURscI~H{mw|GBD^5>aWr6T0q?eU*!es)AkjAZY&_^!kWRgbq>79i{0P-tM1}M~VE~IVr>99#%-s6*C#4rh#k^{1uAzHJu{Y-7{=~~%pHdq1#?g^q@X@!l4(OQCe#02IR?xPXyC7oTsY@i+&-ui=bN73^6)aZ0HerOx7%GScdA<+(Yiv);__C%3C7KYcF}x}F7&^N#wc6{DB0N~#8oX|4(OEaEnbmUkqAd!Cnwdv@jB1Hi&8$q93w3i`7qG%#F_7TL$VZrK=p9<55A3BNf(R4V!;nMP00C!Z3WWhE{i3&0$-t5Az+a`5lZ;Ipi|9di%egdEoIb{CUAvK7;dnvDHLJEiXjx3zr=gx(?0lLG=)*Y4W6aQ(pJIJvrNApww&Y|=NwF7I5H9ybI+=j($KG@pc%<Q3-AfGYW+VXrIB!^nW<bXfr2x{F$oc5TTFNrXSW7`sy0@{8G9S<sV>U<K_>q@u1Z`@!tf#d$fuAhoEIFp7q2yRP=rbDHb|jX~l-kQ~^cf;(ldsBY7P*Dt+?7X&Jb*fB+f$j`PLf9ws1}xL{$aIRt)49#rY8?$KE0si%ll%dlQoi*8W;zslv$GNK(}Tb)<KH7lraD~o!)9n10|#~j^X`4ofrUAT8AX7cId}p!pTV$awU{ks$yLVuaG)-$&#WI2cC_p5f4Ox%T~hWoO=#W4C5-FNtyl55W5W_oq?@u0W?O~jVMZSWxy&;vXv08i8wHn9fyqS*r*Qmwk%52v8IB`!4jvF<&<XN3hGP!O3|1pd%Xs~mh)+eB-tiLO)lSLo%M2Y-Fk*vyUAjv7L+R`Q0+J0mOSjO6{vBUE}|yxBgMQ<Z;%}`S5s;Zr_nDdxXYCUsCIKG2UN&~lExuhMniFcdaO+wmQbsqWm`1>R!RVX&gGNZt|YG{mF(zi-BEk<)pVWqwHF3(nYaby>od6Q5W&mW4Va`KO>y>-QKdlt?F4S&1R{^X`^D$vAE%ekfuM4rXK4K-5+2HJsB(Gv1R>$+HhFz`n~Sr{=RQaKzeyMXLVO*siv*5#B!bcm-C3n_5}O>1Z^K#sy6$q*DdUEQi`7}>8vlB!PpxF^#vZn967XfB;5Y7Me;a<bt&-YdQINGVN-avRmla#6>>J=r@86Nr<w>Btxtw#T4~aCl=?M}(QAqiM!&>b3z5~@z%x582Y!CM6f23AX_1ip&HV^iwPAf<}^03(7<N^v>Sp*O6n+HUuyY{wamq9~oFE8e38m7Vge1y1iuyc2IMzvcU>#Z`{1+HM(;LdkvXS-Af+nR*Yc!FWh-9<sAiSr>;OPFm@)tW?VpdWTmqo9}!c5dJPlLdnJB2Xg-JG4L=M}c@L2!h?27m)E9d><>2^lU-=$H^cLN5w9vg1WjFt2UV31>g{|Msa$&b((pzftSorCt(&Yz@c-b$BJ+(ho>G6@XjL^isa{hnL&LGW@$wFe6hDJ_im75WbI9MmXS{U?_d9;I4Al$VeWe~&JD5mVFM24G=a<xw4xd`XSEy}d*O&58VSd9(=%9XpBpa3VizT!e`mdQ@sS*;J%xp{w>@ieb$0MjJHniu{PNl5C#>?G&qbv{c7^tUc?=YfIF7C^KAa*U$c3Wc$@~LUB42Hwk`ro5KQCmalyeu#wrqn)JorO+yTfu^X=Fn8a{%K_4k<s2;j=PdhVqwxfBDNfqmpYi53)RubT;?PX9_zsri9ThWoZ8m)#|H@pO%asjr>q1PiC}(Pkl5&^Al=6i-1%l`H2edm(M(jA869WW^(7iOe(KV8-x3I4)!2x2bOqQn7lR=g*@xm=m{J1Eg+JGT~sp0m6tsWry^#uY~%OEKhba6I`Z2eF*1$Gd`6)YGW?qyJbC~fFyY6i8DLP(X>$2Is#@Q=Up?L@Ia5nijYwU|=WRe#3mvsX59Pnqp$C_GbqP*Kxg8I*zdBs&FDXV_;;)3|ypO|tozn%n_N-k&{83)wp|YGucC-8^^}S36SPCPR<g3FXt*dnmZxM9~xTVm60w{um#<{0;WwjhAZEo76H893vvQ=etNhc)-1k`RA2izDbXu(73m3$3g0irlR*i$<0;3*EkGqCme@>k-|-hT59lHV7f$pi2iG@cqJ1-9RiS83tMBa=e7w_QQsX8WTv1LUwxDa|~xEbAq)L|sDbY5j--$m>@Z|193Cc0sV-L%>_|aU4hNvL%<FCrW{kfaB=?47&h76K7*fyr=NyT=9E@Ms^g}n@myei?{Cgxw4HG?Y|mD?`<yITvH3;HFdHC7d^0~z7wV9LJ$63gk-MLtaSb+p?}P_rK9}7H-@MVhJwf^i!$g$mF+Bip94oY+vKGVL6t;|Nnfe6jUvz0i~k~hMw>6mvt^*l3Mil|z3fl^@LR8imLN8)Ql)CgDmx`%wZ^Jm!nh!(=Z8<9J{IzXQqEc_BQC!^dZ6H`B%T83Pf8wx-*bRW#DuG6%({A3n4CmenovZf`TclaOi-8=K<K9upT8-^1SW;E<#78WD$z-r6#yzu$RtS7q1z?ffKk~LWdQmzOrYyId@+px)pz-FWgmje{|0SY1lbPf1d7Co=;DdHOBl5+2|@78Zpg%QRJx8Y|7A+NL%=~?#H3x8cnRKZ6nG>N(_vb)mfg8cfPbe&48P?lr#Qogro29diRKjXdh`PZs3n;|>gD=mTs|e5{;Q(t@DhUI%m2QB!GH0e$!@Dr8e9MOpM#G$`4xfbSR+r<w6L@?-SwbVsg~+f`6CEPdT|3N3Q<dniDwh-rVtY5ITyXwW`tjIgWN7)+RnxX`zc(MRfq2zb6CU^%#a{>^HM`4<I7)S>tOAKxB|3xHzY!=eN`z-PHQs*qTY<c-h(PjiJ7(L*%*2$RMW*jOo>CEDEAMze&KC%!90d*0huN-RqC!PC$a28vwk6cpvKJ~-d;liI!wEo3d95HXH*A9b(&6MO{QkJP2AH}En2)*L(Ycb@eXe6@($WQ?*@zyn<G_ob#R52RL1leN~b4Z^8Ke7UxuT|)lHB2RRv_a#0@*9A<!L5&MxXYE?Y1%$4)VwDQj$~C(2z~GuBO%kD2EVY#bH7DXFKf!=gh!I{JY#nTS!2y)kEU&KC=#Y|70t%RifYa7j+haF&DY%){x1;>|pIoz17^EEsW5jH>3E!oGq0VU5Ajp?ai3O7|MD-c?VHvfCB^B~u>Dp>L$ct8@{BZvuQ}RH}MRV;_7IR$U`@ejLqSrP_{hpTZ(~QOcCA(a@F4YV4!O?t_P~zIN~F!v}m{b_mSOq&p|xxlP`$B(KeS^6=h+*B?4lN-9Xtr+H=3YF%yZ0BATq#H7DWl%uG7?5_yX^o9UXI&NPPhM{AKT=l(JUD;PxW3K^YxHJtgs{N*53@4@mMrXEHWZ&%TcUgNht!>JV4&)nF)a>?680G2I*Ep&o$t2A~Dg!B!91iX-8?d&@CWkDR!F~C<B3Y~2RxNh)O(7z+b<o{8Wd?_3fY1tkFpjtI-<|KOU(M@4{D}k%Az5YQ$PkP*>ASA5wdUGL>&^+DN1OG=7JPs<*WEO&Jy;cc>4I$?$KJZdb=t+cg7Au3cJn^#Dn>2858t^2(j)_?XF-b_QJ@3E_wta9TYIxQSXYhe?Wmj9Rs&kCNBxzz$`y*P76m|12@=jQx0@GBiCAylp%rnv9A-a{J_yC$zA1dmNos13o&gI=Y*TyxijdnK8rsXb1VSEmwnny%Jdy4v5sf>*D*@%l(aeREllmuEx}D5%h%1~sF5F9N?X~Ip>cfgpIrt#zOBK4xq;FKfvZgUE$SvPA-{qt{Cvb$90zb+v()t3JcWN0BWEE45m)A4~D}_YEqeG4vCV0v>)Ic2%7g^IhI$MEaB`^ve4xFijR%2lB!j#ID9IZL4sM3|oT1~EBwke+8X<y&!%eN`}r1xU41!Cyrt^Fkkt(|O=$U)k^ce6FW9=%qbgf7(Xai}(_j>M8X9yeHJ%?i36i+U3TFOPzdOS4kXr`f!_oU)SDbt1sN)^{<oCoShi_0--AeHFj~Kxz~puXT@>B^@x|EOAM+uSs2hMc=cX;b|^u^!`!H^4D(^zi}v~f>>8)DV*7y9X-kcwHswdsNHHLhniih#S|a4L}^9_5ZC&LVH={%EV#&(Bwfs)z6#8eWztqxIh^7YhNH0XCymY}y0hU35IvUJ1vZ)2DBy<vk#^DI<Qi!oE`@c-zv>P!C#$+wfDchEJQ}RT?~gU<3TKs2Bdn{7x$7jSeLAElp|GBeqH)!jr)7rw{ookKnz~s#K)6Q_8mQ(q0`t~8_g;JS^e@~e4<Enz<f)80wU9QWKI`5)4<0>j4%DB1wNLb#1*<6(5kaf)MHI>-8V8|6frf0duBp44vT5S{Z3}0W=v*zJhJKJ|vBg!@ih!ixJKfbTo9DVQYYnl<PeaT<?^3k7a?n+c^}PjrZZVUW60ToLPl9scuMgV5kTk&e>nP=44@%$FXvtq$;$td=D5L01Uk1`$?>_v2`|!z=H=n%Cg1MF=>Xya{Q5hM&sGKT$XB{K2Hg-?E)D6ni=hpvcBJ`5@5k4)U>ILe0(U>j*33<n*aXXCqbCR?n<-$VxtgXP`50IH><r(D8s(@EUf?b^X-WfjpK`omlxFV$v_6%+tO_dB*Ut%mLM}w|vots6EOKHw-{?}FGE7((qW#<w><d5rbEm|QN3IrU#P;#SXkN6wZ${Mx(h<YOBT536KE076XurSEr4rloAeJrnTQJZU)Ik?L{T4^e))dpWzj$3~zz{-Zp|2ja&LzSOq=V5rPzuQ~JuPu{ZwtXXhvDe4dpN7@#>oqB3)3eP#t8neta_VjwPF!r{hU*X4$^_Syuo3O3o4J4eExJ@_>eDY$V49@-*Elk+49LIC;oQr{d`I&!_w)kP%<<O|_=|p*E^xZ8ZOo+L2*}GRJ!@+VQe(-FrC?$M1C<ry)h?IM!9WsXHI;wSWc|T!eX+XvH5(0a)c0%jd6G#;*fT6_IN|4(Mq$tx-nz+L-DWfHh&OIDR|9m@(Skp=gDksTYH?lu+S+xA{;q2c)HiU%jE(;TmXyKG',
 'export_snapshot.py': 'c-oCtOK;;g5WXAme_-7MDa2NSpt}fM_)wrJ(4yD|Xto#EAQ*8Jb62mB)Fy6_|K1srlI?gmtqwNDdGec?kDAS9*H2w*JyWb`>rNGZ&VK!Q(OGa{znJD8w{6$EAK7Psj#+r<!FlGbD!?(*;0`Kicg`Gi$|?`lE}zY2(}}60Uwe=H4#thGa@)$>jqjga+e{~=ZEMykPa>8iH~2c8jEdb(S=&A`<xuhUZjM1PH6Nr8WHp^=C^0>yC55O2dC!*93B%`m=Df{aR~cWK2As&4s$9#$j$<FsE-x;x&Oe@ATwJ}wQ#zx8eNmOs?6*Jv_&~-44elTsd35<`{5G5*Z#!rNG)1dXX~lbAF0Ob+nV}u1I^{r*1>f6-ttrF2YL#|^^2u1)T1G2^I9f4NhMk;(cOu*D4_PbIh~a1fZOc|8^7&8pzOSM2H$fsaxB@+R(>5z9bz4Z89h&A!YpIfr;EP4udp-}PHx~5j*7lGLP|IPYWEcL(65)(;v_vu0QP6gd?@8j?-WGtug94HEIpeY6a#a-2dC++f@?-~39Td2Wmd|$0jpmyrvjHv}unK-(qB0`Zw0?+1h*Dvahm3u<3L8<J9acpd2kbAZ?lm@-75uvH{4R|GInq5gN`c*u!{UbAN&}T9JOt%6XS}x+{gtLsl}s-lI}XAEo(NX1TPh!|a0AE6x0YVI{;o1^3s?$N+vgS^Zq=itW+PQvyREu>_kQqu@H%E1DsA+bEe;Lkx_<r)JA^R52QP3LTMrk6W<1f$A-oo$OOVY;ZB?oIPAg{7N2~{RW`N_kCI|RpT~PO-$AR)Y>Adp2;~T06AF9p?Whx#j2BbDa-!G4`IIY!(wpk2r$P0wpMV=ZCpR!0?lt;M4K314s5LuRQ;Ypi&<UA1?CaDYnZ~ilzThtD5SZ!Rbe6byJaK#pC6~-rzYq8#Yqgu>KWE%he2MmtsQdL$65~9qY!ayj-Ch=12^^O!zi$B^$3Pb&U$dkfH(I%;Qbyq=5_m-A9aklJsl!2gbnx`YJ;@?0j5BZ}tm_o1@;djoCXFZWO_A6aNc=tFWC;pG~W5r&vLn4L2HG(WS&p)+Dgn>(SUw2N}IfKUa7Fo#^#;krpl1FkqWBl)i?;Dho&q(zZWv4*_;wU=!Lmt>F9^~sM$a3Efm}DahB^T5`80t-?xk>k!ZTm(qsO3)@$CTe@&4kPU3Po4h?{5XdLi80e_%pIp5$*%Wmw_}BWY~(*?>fL;9vYA+XBa*i2cP;*rVk1n>?NZM<do48f->YIDIfU-tietkZ5R*7dt0S66;PJXFE4u@YE&JrQgMva{Iq@th-u|%JQKp?B8v@hxAQRGaeJWFDfVVfx|@JoP)(STKf)|=$MQA(1o<K1yd1YagfBH&tHzWl4`(<Yfy34hDb4ZF8q2o2{C0nLNCaOoC1wa;UnP^04kTsZ6e;Pxl9DH+;>X8y@-L$=LTC',
 'postprocess_snapshot.py': 'c-p;MOOND85x)1YNScG~!Q<{7Ew36C(kwfxMJy~DX1UCctSmdLE4N&>Co|pMwXK%8aYP7l;<BUB3ULDo390A6VG#eL`6Wc;!%jb{x>^hLrTxf^j913@Ww?%yk3U^53tlopo1Ci?c3D?K5nAR%Fqt)s)HGYuGhUv>gl!fq&sk1*Nz{swk3S+!T~stDnV=U1DJ#VmRkcozkB^V|MoU^zxhnX=-F_jf@@Ofl4aimag4h?ofLk|6t{TOQqobpoEkO;Mr#LOd8PV*Bz@K2M5v7)vpaL?)`DR3-XDD{5LRMx~4yw<CMlB!yAc!I>^g?jOI;k-^r$xi8aC}1Mq?4$hDu&8Yv4vOoG`ZkvmC9zh<d=F_Vy?+|NRX&a9So)=W=*Y^);n)DwG8#b;M!6)f~8buoX<a^g@k9k%vq`Ck0KQLc^TlSfjxsI1gi_0F{`yPb;%2sVibJ;<@}6E1(w+GByvR`efN8z5r~#~<|kFn$}kX%AR<%}sldS-BM1MMl^~0v%GQ_`V4?^Mx>@9OW=HCHgzud`dY?QX_%DjdA_$^RLZ?yE)Hziw)bb)f)b{;~U7D*fYV#wjL@teqRih5F5OvmJ29+hG9LqzKK`4Nnkibd}qU5Zo7Gdxp!I(h;bz1i|))`RJ4NG*8Sm;6o!CM5MpW0YWn7UO=Cb%WQwr~)w=rA9gGr^bJRBvH4AtqVXluBA{^>c7Vx0QOf>erCTn+@iWu~X0EOA_jr$r)2&Age}XEZxw$h8#@wqMYSmBA2-L$#c=LUQofy-h8MLDHImH1+<M!-d~7h1<RD4@U6Ku^F{ac<({cowW$lXXRa4qLAhk)8I0f?_Ed;UguxT-#%%Skzff}d?sY~syj+og{_ghepU6)>Jo)U|C*=IyS32Qf{>2H}nFNCvnF2czByhIVh-Mk9wMJw<C#RlPIuG<M2yitD<EWn**-)STP~$)dvUvx=EBO+WXb@&M)p#nevStIdNA^9{={53lx$TQsvu!V;ddI;-+;CxUHvYAeQ=4NPHgpXD#Kbu)7n7i-;#I>`DnJ#;Ai?V{b>KBxQ%t^$qC2Ac)F9q;SQoc9Z%T5u;>2mFL#!8ihzlh_S5R>GYx^M6MBT{5Dvb)-Ct6Rx^wpxOmJqML|FH^4n09p?=?5rT-?j4;74QNkn(*R)_8rC({cgr2T&5^5j4)94nN$?22mDJV|B_*hwIXJ9U6H(c<;A%xPexEQm}-h_uT1Xn`!e_##GL5mjdRF)_3m%w8T`9_`^VaPWJa#gw(FoP_pcfbNQnu)HQUJL^!(>l=`b7^$TRi@>}yi6Gn#E<vZSDUjz)m-1um+q0k29BAl6JsvS1KKMyiFPyo3@5w1y!BbHHQGX?ATqCAOWx=vy|m1?x&G@Kwg7OmUFS2hY<GISd?^yo3QIAh{M6L*>uZ-98OIxV`x+stq3eGyLySI_q6{^`<7p?c3ix#&I{Vw{Uawnv>1#+pjb7{eS=Z_diVfDj6kgfVJMHEVKnRuwHe844~9h%Dsi%=9D+l7OAP32U!2aH`navRrT&G;HtMbU#r8+HKh)sr?q+!n0v3_p+8(b1X_e_49YZk<8;b4H?o7Cmv=(X|I$3r-t89WT--GM#l1m&;Os}l)%}s(_}3Gl!Jso6ZeRBBpz}jaJXc}aw;ql=h!_=@UH<x+Gt~IXmX-DG%|8@*)j(nI?T^F!=gNHC>K+DtNCiz>o@Ab<t|-Q6DMjln4&YdQ-wl~`ruhq}h*~_LZ|RN-%>9EvhZ#C7-3G)!9pv2QWT^qreD?-UhIFgp^AFrr?}L+bMCSYYIa=OYdH1gQ@rq^Zy5i+nJ0fXg6hBPAoh;k8V1mQ_h@9#J^)LWCIN8AD!JvDXJ_wPm978YM%k~NOIv}S;bxTLl1tk3Rms+8i{0h&y_WGx6Yp<SfYxnSp0#3DPl)2ZZO%Sr5zdc`Z*wPKu!rPm_XNYgV1!EVxk#MiUME53OQA;;?2?unp4{F-z2Qe`aIQMFkp%#GYtkxU^4(h0z&qo|7xN-<OvvS%EY(Z~E?CF01hW7cjW9;6V1M1-pmcCZrwNnlb=b^r`XH2gp(mNgQVlBoItam^)vA0C6DN}E*ai1xTpu#1~Aj}<E`wZfWEhR~T@SAoL6S~|&PXpW#<6F~p5^M=4gB1+Zj;n-?D-(}N$ske9^u2`5g<}|A7h6ys31Hwxvl<2mMwMVp`K&o^Ut9H4S<+grDwWh*EYx;7Ts?dUBjKXer$;?T#PL-FN~&#Y1Pn>zfT>`TK^>p{x6t-_s=Bqzr(cDvDmKMYm4KHmd*~M|lQBl1XN&Q)j%={YzZUWO-t~N{?@X!%F6Zu3#;qB>eTl~O7_3v(Bwuhh2J~GuFv)KA-8S8;6!+?Y%jq@_G3k|-IhwVDcuZ57`3T7d5VlL2Pj<#adfW)uQXlN+BW|!`l_@?nUTmuA72P+8Hc@s!G^Wgfn18f;TUZqRfJ*jMU*d^%2#k{fH${JdPd)@JgDVB#3!R%RO41amTAE%1X@dvXzETLgrp>`iS`=R8^HAfCQ*MhF1|H031vtkyrODxoO007(G{o{eMI6=_s)FM$RbW1Jf7m&kjjPgaYU}CPNt5H0uxOFyoSv1S8qefGfDPyYYy>*%iYzK<6jPC{_&Ec=p92ojCBO}##2Sjd0xjV&aa}9j0nA$R+3O#)#`?Gvn@js9+Kcl&@G(2<e7fy-ch6xDON*u#xw_uhu)smKVWCg=ct6oQu*o?XqMhg{u;V9#lgtjtp3L?M-eY9$YugMOOg=)J3hG0Fb)U~k2Qp)S-qm$%h}_Y*+u#n{8ppeKpY^eM-pT$pd1gM)Z5lY^g2+l|6}dp3WS9cEfI49)5$;n3FYwGxy0paeJXBTD)GFDNU6wFRCD@>l4Vv^ZIXx)TM$~_&0$rO$R2S&F7cYCKbN5xp>9^}cPJe-s$)<GR)Sl7Z=fJ_?+S>tU+GNkpS3QBxD8QoZ@Mu5|4&9UHd!>|P7J1c0Vb(gZ$M|N@8!bcBCGyDM40<=0;c47C{b4t55_3C<goT<-C`=MK*4l3S`P>_7xIIQn3pORC?M;FD?LvV?gA}M_(MTx5uuh8*frg0TI{=Mt{t!K2$QV$99-^QnJ%nkcJ$PU|?0GVnwL5%#n)BWa4|MITPF06v@OBJof%{01n(qTM^5U*94KEFovE>Jw5Ig!43`!DgC9~N6LuR|<zwtTKZc+EMfppkObN7+`qU#Vm)$$-o5U-C+G5oN@cVa(-OZXZs9dHw>stwPOy*$=-!RL1K*;=|EF`RR%516eQW006&6F|2F>~zhdBOelWv$Y9Ju;VavaMw20RI-DOZ{$Yjk9kW%P6!Y|5TZX{UngJkIvD-p!5^D+gVPR5CXU66h8OMG+x(~_+0Zro%)#h@t_#~$un6JWK>Vvpfs4f)*MZcAPcZP40obtZL*2WK`U|zcez4M`o%DJ3e{%I5+}XIL=K}rX5p+V7G$s5jXB}?1rk828_A(5%<|v0I+xezpHXQ4>Y^<}6{sZF4s&o',
 'priority_probe_snapshot.py': 'c-qZ<%aYqjlHi?Rfl=EUKo!YkrmA|>l&CSaq~4OXT9QdwRlSy#0|QAgiAoY+0iaS@IeFgm9{09pV{@Cs9`-bS*h8=Ff7G9>yN7!q0tkRiNxjyxvPuGor-z4!zvJG$d+#^VdhKm@vvo2bZljCXi{|sVD7<;L*+zL%Wa-44<=Isc=iVxg7I9v9N$S0ef8NDKSq!{vTP9f=t-bg0I!=?_W_0h~z1}j<Hqb^o-{pCnmZRma+~qM;-E6bG^iEefwDmU2)-OKB^Ie(c11~B<>~tNM@j}-x<B#P!nMu8Lw_fwx9>1EG*W0*|r;A^bt^EE81~0!iQMr=eS)rQW!aJkww?>gXmS4qcS0-!yw3}`73<0FyT<f1@veDH_nqZP%sIGST8hRQ5M2lGZgLgk=NjktEi+Gtw7c%ShI-Vm?HMpK-JD4-T2+Bpfb-769Wv}=Bi+Ar|zIp9UJ%5`gS)P>FLm1sG9)5V}_r87c^f?rOuQxpp{`pT~VYj6}@q7f|(W>062hny53m;);M<4OcS0C$dH|s|~PsgL*4elS^r(X~Kf!W~jRg_-r0CTYAUsl6suLr-ZxJsPbHNM+k<bXoMmuUg3p2tH$BdmK2rEYt@9?a28<MJxYFT*&`vpl>^(nT=OqpLN-ApSV-PpG{`Tt>+nW>}PYfUigE>?+QKK7CD=UQ#4!QAX)J#?J$f+Zg5C?Fin+gulh;{rguhus3cS|K%|tZR+_&u~sATKc(SkS(?Ino&t7cyVBA8TonjQRpOI6c?a0m-UH%dqxmSk4rnhk7?)Y@(R1LZ(I$r7nyq)(x|;gH^Dm&_Vv#55GV_*DC0^iL^~)<WkE(o#{k~%ei?pc05Jt7hV6P_y0|1&ch6AJV=j*J1rujPz#k)$%6-)xV+QPhJqed%Wp<bw3s`@sXgr?&1&~{YePg$Y%)v*@F(I<mV?Tq;f2zH!a#354afJySen`Mh@Zd=GS?EFjF5fkqNE%4<4e!x_@%qS`2O%e2OsUnqyYS`A<<am7YwRhjcKj+rQnrGQ%0{sKA+Lly2TL*bOD&i=guYziRGyic=CgnQ*m(_9q$JyCGKRLhu<j3N^YM}5oyQ1`Fu@|S&3{VKZJU;sI82&r~yrLfaR*!kS&Y}hM_=jj+#IgthfFq#kdyl<`$G<xP%wBjif05018$5edZCYV!R1XmUVcB5juQNsOMyijhFyf*0dApA0Mq?(f6Zu{P#T2@Y6jgsM^XrM#b(zO8Oa=>$uqse7I55;ytb!i^&j3J3=skaghem*PM<4M73;d%ata5aOWIgDgrCAvuN)~XcPA0`XPqt;B=2`nvUa#Z+IqcRE_Ph=3PU|p7DDyxN(8niZU0h+g7knSBcQFysftpoJ!+eXUUS21zu0BrlU2N_GLGKYzp1_|khnu8KE}~Kq9$(XYZwjE%JHy8{vyjy4los?c;pLN`5CbD`lOdwCU<j4p#yM=C4XlqxM;Dy6-iJ6(mV8cj)#bjTvV=|)#rb)c<Z%tlEMW9`yXU|d#KkHsfaQRy!7?uAD?9~v0MZaBKq?dkxUT0wN~0V|@hWEeY?96`Y*X)Q1+Tp=lCk^)&gR?G)3+YC4(BMqmiRYp5lVeXvK<l<?sg=@Ft;TTpNis09o}^KcY7W5fpY<tCD32Ff@%gRY6(M81Z(2qDO19ME`<l)Wna-QbvO~&uMq*}abDuzXjiWAZ<)h#hH?7AXMzT+|B(g=j90Y5fx@V;AA2-htT9Set2uaVq;G*}MCS{GWf8A6z>sjP1#CQ91?x7T@{@pQwA(IVQGy!8oMo8h8+iBc-}v9b3J;$mf&g<>Vau)Ef-Doaz>jtr29J~xXxU{5RJw=)oIO^Anx0JsLR#7Q8qOu89=ODU4{|tFG^1eD;5G+FMcFV{QBsD`ChQ|1xE0r^H%|Nkpn#lov{)x;3~~qTOP~?lO#z+4+Q>-`$QUlyi-2mK`M|3C=L6bYVF5cSE!Wpmq*{I2!CYyT6{S@}%{0xfpk+MmN1F_2ah4`?UUVEaeA`4HgA-Ui8I$)_#ZmwP&o-l7njjqt#shEkO<zxIlmhJ!qml?EPGR7U``!?{)XXsT@s&3^_MS}j^i`P(!aQmMUI_36-UH~=IUYUFfPO5ptMq6V%`da%lE{9%D1b(7q9jFP!hDZ6+bu4HO)|)Nb!=PR(2@aF&C*3dUF9I|ECPHz@E)k~6V!@z9B=7;-#jJg)DUt>1AxV!XZgah`bC5iCP=Nj(xjmSkD0Imj~({Vz@p@OCdC$cbr7?%vY6702A=9JTqOC_SHBCtBFpbr5dhI6>6!;ZSP$zNA=jBi7b##&GB4<qM(PbsY=Q6slElDWU22j%i!YM24T<!cPPyDyJQ)TSj!S5tzyT<J4#z2=&QvgMSI&`O3lBRzJI4o@8_b)Y9B_eS7dpntvplEkhnsAH9|&GRn_=Q}I{B70Sb8yl666HxCsY~<;&DTgX?<6>yb3oyc7ihGHcnCKZf_nS7Q(WFG?azNNQ6q=!F@1sWTgcPmINl=#h*jI??9R@J`wRGM4|}$AK3keH3o|22K#g!UBif8gREwKZa`tu0AhnHbu=vqe#Xm4n^dN+%55`PE+DrGa{We&%pwYoJ;T4m5w>*R>`&MN>tuyPU-b{i@FE7*77*{|w(o8;N(vi12wb7G8pOvHQRQz1Y^39UbD#j-`aU#C>!f`Y4mbq9h_s<JfGo{Bi=qM;R}t_yKuu;bj6U{=g8B}4e^3Z~L#R6ultyMrO$y%eFM5YX%>94cYh4oA;^!mK6TyB{CtY(0Oj_Egm9-FXo`O2ag~FXUiu~%X;y;xXY~m)<pbRB+4T9q_i?#g6N$BtLk2cBF+%TWpBA-T6eJTk~Oyx)G77H3m2*#loqKg=KZ{90u40Nto%?;dYQJ{#k1TDN!IMPx~MH6>6`DQ#mA9z{;0ved8D5j{aRjRr$L@iSo9ssLK3pm}U7<AxP0?O%i3j#480&BL#L!rrpG&cB9@LIDbHIg7;OX&C}gl?m8fY*_d>#?MvOyV{UR9JS+On0K@Bu~yNbXv%mkP{Kw__#?C@$=`h+HRoimv>t$^PIV^($!T1BnW5?gX<q!7Lv53fTDjEwkwnr9MW%mVwf0^Q~j&o26VNJNX1d(yE_3T<ha49xA5?HJdUiL<_}26&UneYyRQ^p{Ibz}r4(pjF|@UtlR;?oEGl9&2c#giBp>4E$edcTZY#lAmE|^6C%{w=0)2*Z6H`<ubZS1ksSm7<R-r2G16iNaa)^Ws4`|U0;&0Tc#H1-F*bk#+sUGW^D|pT$IJO;}3V<p^wH|=&Eot}Q#gRU9``(eaymupPX#&f78{P<f7s&;bQVPV|(Zw(K8m&vosLVFNUeB(}xCp2pR82$U(*1xeX(Keq0Za>ObKq@|98SmmJu4%POy$Q2Y<t_RNIsHa0WXVHbn?w_r>=pwv|T4<YQ<?yz8*gucwe9Vj%08^Oz*M&{Mh^Lc-(8-b9S5E6mQbDd;b<x@Bl*eSv!9Q2BFJdJ9TqyEk^th5r1mwWuNhuIOQ$jqh10}-4J-ssjyug8|}03rL*BWInV&I-;0LESXKkS9*jXIcrZS}|2@F}{UiSGq0*^aka#7;z_GTZp#gxvGO^{i;z+rXv#Npfeg^~2vUtUT-}p$0EjGW*;8FU7XJbj>gZkkhKb7b!b$bUPr&hirx$iPj)!~es&q;01K7eObr%5|aVE)GYBdCH%RYD4EI;0f$kBgE3hayH3QobvFi&C^C#pn)FFaw=hp*<!z2;j-1dC8Pf%9*<a`2Z1!6@I8zXp6M5@0v?@k7o-^bg!E=slX5ROfsuZ^(&JxX(55QAXWL^!JPS0q1X+hR%n=G62ok%^f0x;zUv%pV@#-~a3xEh)?YUSqI%#zs%$h>zle*jp}<<$y@TEGlb5-Jrb4r*{gHNm-bHH|AinMo81<(<_lB%r&vkAu*U?2N1Rziy*d=Y!jx(%jj5_oWtb1E0z^w29TJ<&!(b1kQPY6nk<N&D)h{t2(A$#VbtV@$Yuc;83^bX?4MMn`Ttx2f0CccUPq_*$5w%%cL^LIcZx3s}DI;uG?{hOwodpq%N;Ki+P&D?GoPR*oPIn}W>5p~)so|5Vpg;UmLax+)39pVsOqHKOoYpxxf<p_)46Rfv%?&=k5-HR9pwj)jIa<>kJU9FfQ(qX8!0TYUB2TEX6vG3EIqo@t`S^NT=^|hDplnV&kUO~Hb^6?{b2#>VEa~01ow{W_`5daHy?XgN&tI*X@XavGzBP>Sjyn?W%m#T>IUhF7RcZyBnIfWFHivtk-O$0U?m_9)dY}82+E@5b<9ZPM(%3L0Ro{wk9?TBkclmaFu1=_>l{8|D>ho`d!gVooF_v|LhFUgMs$9LrU0=>{4nVQc6ZPMj6sR~u!A+$~`LM|{s(MGKuygxfD6nX#@SuuWI`3`Y8hmpXqmKs{A9&QD$YS<&1u5I;;K9CwwP#Nvx3X@e8i-wR_6IP%T7c9sBFs%uIGG&nf`z`|6wKi~o!0EAgggOa8&NRNF1yR&9%dbZU@#gJ9Ba(@(VS+)R7N>#(un6mH7WnrWZw^Bjb|omf#*xQp@qp#JOn_ZPNTNyS>|C2))uI{8S9Q&zXocOfNn6Q9ug2UFKi^zbT39t8;bP6k>b@v7mQx$EnmB>|Ss1~`q~gG!?Y>YQ^#(DxPiSHdU|`#R?u;ZapC5hq?v=Mm3UEE0uS}e%kBPZ*8iT<BAZam+HTgOR+r;FEIclpLQvx7SdE8gL_Lo}~l+=W>RuoYSE-c+aJ2k3mpi9*T5>9Z2iBK&B<~kup2QLJ)q|g^Mv9Q?+*!QwHuuaSbAyP{Ejtr(8R6{^b&det@IgrCR+Sr}5wLyIvp_64$w=IzA$ed&4toEc~h%kK!Oc56m!vvy|hW+z9fnpUEtD_?MC2qsXn%HKdPD5TME1TtX`1#oL8}F@hO9EaU?4zQJ5z8Jpm}ZZ>3`D;C3MZ<0f^`hUweS`|l-J~h!0T>^dE__=j!4=}7$~2SNnIK>bnsMSK*ZW1%`ZtlP1XyfnpD&+ui3p)4I2#c+Sx4{>D?wrD!|Jk8>&T>1TNthsE4t3Bh}xx87<j3%F*-)ZKxyb&JFO+meI8AYav~#yLutpbJi^^I#FMvPn{IhXn^^zMh|4*d#Hh}vgBUxLLF)_LrNj-v{8$oh7C*^lxnI*N~*SyC8`!tsL5x-x0}A~RYI#}udAJ<^)A(Ob1RPM{|$D-z0axYzzcB4@pkXj%y#T&`(s@?cSvG&27NDcWy@lQ`<yaPh~UKysr?CP%K<#~*Weo*gW+X`{u>T5!Y9Bs+Xm!kho;Pa?{90U_!Q-fc~WV;vtn^<9<Jw@nXkOCLJG~v!zG-7=-=iyrhB+D=l*^1u2P(Lh|1ZiDNps<)YTkZ<aWsF(vtVut%L?<srt)%b%Z-|;aU@Otqsfa%N^}St()M^iik?GY3DUWawL~Gk^A(rB&bGV*+`j8@C`FCXr{4OpjDruq|kxt&C-%6u}>*U00r4=0geK!279v7bUY>H$q^$w>Y&)H3X&Z-6D&vtFfWzGHO<oDGAiL<XT<Z+4H?8gR1X$f(;=!faAml%B#lY|T^Tya0|9ABC4i6}GX;LnwnU2tNE$T^-)hgP1~GmIP68mb7b112=#)wfF^6VJCo(p(EF9gqBWn_lS%QR_#yCaQY?Qq99gT~#25W_7=;pX&SbESdk&Gs;U;_V{T5y1_uE`4*(u%1{XXkeSvM(BLqlccsY*jsBR0!Acb)-i#2nl)Z>yQeY7)<_Hy&gI5c%=P(n`ale#Wm`_g{YNM)Ndg3V$|XWbjd&`*Vmwm-)&%nAYZWMn#7z4<`Mu9HZx=y;tG3l&x9c;M9VM)hwt@rz+R@X(V*aBx5iDn&Mx4@1yVM(1zGgXLlvz4-j_AT!2&S!2aILeCrDL=wO-Cc%8T47HW*(3MKHJ>9PUVu79Wfu({t70v~yZ^lR7>Y`AU4jQ0FIDBsj@&Iq2QkuB?za#$Ctpq(N=Of|Q4sRIAro44fk_(f?0^VMk!!yo5s40xRserUf>Zk0ue2QggA}Yzwe(Dk!01(gd!rK_XxDi!R%huVF3K4B&G_qkK_M-K{aepo;RYHsRfAq4w)Ax0<MbJgg~2Q(W!Pm;Np8qyo;CFe>IrqMX>#X20GQE5j^OEu)QBHkhl$WT$(zSF6+Qz@tbJ9XuJ8g}ixK+WH!eU^V@Ny-&WHddJQ00?#w`<GiI<9sFkv@@=Zqz#eo7u@qpt-_x8rrNx*8&FwiA`c0p1gh^9KF-byW2zT37WE@71dc6|<Ax^E#O{0Qe=EQH%h>$GI52T7cq)p|)tm_eTKafqQof!^l3Qu#hEsN_IPz+&Sm<8|!q$L3AcHm%Zm|-+k$C*1dCwsaDRZyLOSc3*5F==Nn6owt%V52RwnY+*%VM^^?{GS?!CY)}>Yt~N1X^oh->=AwJ7)Oz^z19z+E>hDR=aVMAJYWPWt{Zu)y_Qq3n4(iAx-xw;V88R@k#;F7rVp&Bre23@d2lj@ieG;->PyVi^!OpZf*`aIY)MmO{RZxUr(RpUND-=ZAcd~oH1EQkQWiY96oMA5j)V(to(_dX!&+<RFd#nfL!H0-fP5WkN66?GFxV<kbsN}zP{mEBz$#w7&iWOjN@PK;9fOz@Z6`Pw9Hyz`9Z<}?R(WD_5>R*|29`7xHGahLY6ACb5o+Dz=WGLN0vtRd!&m*#7nzAMfuQD#ZU)JL^q>b_y?2I>^Q`A99DG%lwRK9={E?hve?((v=VYpQDTDt89bd8_QJ6#*_MtN-(u6GS7n*>In0oAuySjZ&g&a`BPSf}zszHu7jwMV|dI7>v{#wUVsr1+iF=`G*%C!q1*c$dJBa%?VN}cux^(Nqm5+oiR-=*n?{6H;MU~g$`AFMw8rr!YURdl$ALx5A#DBU9D9T4Sm6<vg`5FzS{gOJ=J!q9PU`CPiIy1c2H0Zu{VnH5;)=s*lv6HiCx`DVl-vOa%{m_^@)DRg7S(P0XtbeUYVb*1QKn=6H*JIO6FQC<f^nNTOJVss}u;8h~SiYhb=(7-1QL21Gr*yL}oAtMVpj9eEZ#ssJaX);QTROS!zsJ%Tz_br)=6vTM?;yNR41C}|XGW8$O1d?FE)voRJmfP^C4UbphP*0C`)qcm=26|weOF%g)k$c7+Tr&<<pPIUn4yTNRT`O!?2JxDZC#7+}qg#c0ZJueDt}d^NrmLfPep%|_E^2MSR<8yBI-79$b4e_rR9RD7DHfO!(8vmK^}fN?s1aO%1K*ZXQB*0t0<u0i)}C82<Y{-70t>dSX$qtiOJ-GgfEv>oHh%7c8sv@HTI!q^C?)FX^Cd^Lhd+TI6SZY5?{B;}cpT2@#NwHOnIG`w0w#?*X0AOU?`>vTcyWLb!r}v9cNF=$O4!F6J-jJXgkgfKG-PATNLE3UhSp@|VRC47MuFUcmX44KO6Tz9G7jt)wrae%{b(`UhFe*Eo|^e?&QX7|)h#B%kJ<{fs9ac$TxpB6P*ugL?$Qt?0OgX3EHsM)>c}%v{(%i<;7A=vtd)pG{FkisnDNF4rRNlcKwt602y*#>pL0;*>_B0LN;X43HMSJmq<r3<&k3ukPOzSt&M@G<U5+I51wAN$2pBvx`y~G!!sA*(0)FEO04PZzrskkvq=E**aCoFPEqKkq$v_*ZkkD}~i<`LC=qaY3;3yEu=dl4kSjorgB+fPfIog4hyDhCMx>xGmYqw1Mi#AsW&=^jwV49;yvaU5@2lhMTeKq>Bly6z-&e}MO#-VE4M;l(Rjid3PX7o9H&Z+>gpn^3zMt*Wb9^WdjW`|CI6R5HLYzgqa!|916GdWa{udJ8v6wd(e@W22AW~ii6U`U1G0qUV0YX=9`+uiI>Nau-tT3=(zK#bsF?*wi65!(&VN=r;LYdbgjgc`g+F35{YV#L~izrtFJegpd^G1|;5WOGmyDxs!6O6Rj=l&&|UG+B)<vJbwShisbv&>>hIm$*`B_q4c6--fBh#+tQA8gwoLhylEu0~Od&;T?%E_DQzaSKJ)sJ}JR0t6VA7i>?KfT((1nLkEfjGyw6f=#UM}5e;NG)|xY{I@SrnJEt@R;KUP^+O2zBll^r<jXlytvh=!;>>oOiY|`!5`S&HKd{H;>FDtKfP$gO8CsQdhIV>*Io%P9)u1SKL*!`uxeY>M<%}h_2VF_8fdINSsDKo-&-#OKuqpBVhHSQc{Jz3O2xuEWlN~zj(^2_k&cIe*5hTiRXpTg~+Xw^GrYijY1bH%@k2O(s@R4Hz#Ri#G7P=dn~K6U1txnmIFAqFa5_PeBya>rg81e^)CBm7b}swD)q1bhtGBL}d%K;}6R<)AyR!x=I{dh{eEWjMJ#q82nQq9tH6q-q9q_WS)k1X_*~5&11rp<PFjgPZzyAPXck*RW=4PNIoZn6<EPd(hR7Ip&e2uJ~(eKLNES6=zC}r!pF{Wn6?BvSXyfh_aZ+$h~UcJ77$$Z1%PbL(59q4t4HzHT>faS-HmfuTS%2xi>bb=}cx+xQjm-57>@Qx>M_Sa|vvVS^7Ko3RG@)#^68E^G-bDY)I6-2c%aijjJQ5+*}-8odV>gj_I`0ccE3yAEmunRlwmmsFL|Zme7!Qti40I@SfdlE`$y_M!F!4hRn_eS}0~26@+0p?71`Kui&Y2b%i9RRRViR61S$T!b-%6qxmXi-4JkKp}6BxzTx%S?>1N}pnAW5_vVN9FW!ZxFW#Sqe|q!$g(-mIP*8TmQ8*ZVmG2XMl2NM46*~&T5Ha#O<ttQkyk$tD0_kAmf#wUT2bDI~8LA~=c@C9f;~=SF&;DcXvyQbrgHe*3r10eS={Go)F5-_qNzP+)@Onvbfr+HJFqh(PbkRN7Lfku*X^1blUM?-Wrl(kd(VZHWg!W7Wf(?KHbjgL0Z3lklm~9|2j=<-c%IN_z?2c|9fjUPCmD34w3Z;`pZH&rE%U$^e>vGhYCTC$)&B8YfsK%v}flb<VrCbE+lQ$a5Zbg9g&wTy_NND`-%r*bV_?mxV3L<x+`G!!68LBbmvg$$Mhk9GfA&Q7$7TUyM@LX`X?rW7LZ6pu@O;`+S9u>XINwL*ofWa!N&T|3oWM7c8Y7?zcMoaqYi^ed`QnbD=7YfS-DF_?&bu)pqR7t?JSNmt<bF5Vr)cA#;8hETwK~Py<SjdcShq~n1@c8^p_WF5WL@R|*+RR;J^ni~8wR3DAABXb{juTyy-8ixjH4nO4(wX1$jYxL#8P^e71tc}-AhK-`&Jho(-3b-|u_u~1?@tjp)YJ0080vbOq1GWO<zX}f;7(@iD82NBk*YD5+6QvdM(oq*1Ln8Gyo*;+j@iJQ;bgU$R5ip)p%Psk&7HTxL~&4>X2HZy^S}Oky7Csg>tFx!UrX<D_1FIe+R62=|M73v-v9Y;|M!3X`j>y5d+D!#`R@sofiJVa{*MGQM*aF9|GOO7_ps;)7zAexC&|+Ttx*v9mBX$Vs-rU5gLq6YL4Sp2nmAN0F_h#Ho4%oCnrK=(qGwQ;11D&_r6BWvvY$r@-0F~@LP2i=Bwq3-84Y+k#FH~uNfyI_evb@i?P3=riI=*##YR{AwhZ<dsa<K|P*)J2dC&zKG;%Dh?s)L7vmJ#$t#**QM3=BA3;EmOOVo`We=~08u&ON@t6VB;idv<bq4qu+?5j(Y2A-LuzMjI5g0e(1Mt4xi$YW$`)t8>BwzN6VQTHmIG13&fEmj)s5~=<}JbO;xet@C$4#=?Ft^1Qc{jMbNJWp{8b0jFQ>9<D2vXSz}tz44n#iiOlCX@s7p;3j%j7h6pC7`{FK#&cMg+{3c6Jj*nGFs*~>MQ5eVRex%<Bl?wJJmIgI-R%zJ{6p*(Qq2{yXIkGMj-fXVr(BSj(}_If+pqKJ6HE46<JhQH-OnMkqnepe%(fLD^3Z?^JOcu;#;k=OG?m#5$h~$OoOz}7u#^YS%4oGvuS_8qNfnvthl5z#ShVobAG@d$-+8T)Z9!4;+$^fa4_GUC|8|h)meLP@aDZ*1`%$IR)v$FfHo3D4B*Pj<?x}uC+u49<xdT}{(=#{_M(5GRInJMu4k-;ejUEv5^o;eFx|QxZPSZtwmdVe*~gn}@L1hkf=wz=4+R_2$?z0rm$euL;CzINL42<<i~#qz(RSaxcj{sHL>=JC0L?-1@YOwzk30%o0T+ISE>8UWgjn`w+;(>YLOny?Jgl!rHG`Xtx3d2=uY-{G6thbiZt+(~)|#7JXI7fM88X#^+e$XHkNu>Ssa<+*GY959rB!SdM~t+X_Zs#kXVfq&s{|6Z!tH$0nkBYYP^2?gAD@*3QS;z`KN%2>1pVzf3su0sQz`i=y7lU6Be}Y+U9Z(C0JREwZPHF7>|$@yFDqwuz4kW;XAXnR+7i6PhEscU>oc7UynQ2H91~N64+Y&bAa@IMQV$P(CqTwfI@u#|)XU*nu8B2lG*+Cr>L1hnk`76%*)C5PD$raN9r7&VLsjgcylYskYyTSEI)XfdzPm*wiI2#4AjtiH4*qG<a(CvxkRu*SS0FRx0#&$l^N_GjV{l4Zq(N0LZM{O>qJf9a=%srCl83Hpt33-{&-5k8FmBvfRzWE%KLtZgpDmYi8L1!@MQ>G&@aq9($`P?14UWAkNtjA416R`K3YQ!hbYmD=ya(j1gxNRX;e;Oe%$Rt`w>0PG(pM3uV}}i!reLm#%Z}uVngJ#vFO?<fV7zNKGenVkUp0nVvNWNu?owrMJ_~cmdd%vknhCiLJ+&fABSe<h)<>`$;x6qEOwJu@wC|t337>!W=9#*#K}*2gQbqoy)9x0m>(1zKX*YxL<lXKymRgMm<JO-2@!L1A|8sc2G|WoRzWwfL`0U%K|D}D<g4{+~+xF9Ht0DhGxpz&ONo#}bJA1XAU6m>nJ%{itw0vEwqvIt8CBl96Y=-roQ6*rW>4H7DVLgXTBS0LHdx4wDXA#~W{R))oAN5sw9(uPGW;Wo4-zAI)^)2Mf=#FBdNO7RI#HZA+&cZ-<HO?3lF3m5|6oQF!&u>jrvDuzw9b4mgZ59K^Q*qUJ8Dpw4<LU7okY!BoMFnCBLQ<{wo929ct*(XEaFx5{iQ2&HsJt8=kH(`S8;xOx0l$M+!*TiBujzi2>WvxlF12TQ0<Pl2lV&6Q>7>z?`JvuqS`-hUA!!C5fx-{?+cy8&`UubX)oS5lw!x7cODT9h)=hl8oUwM#9ISpjZnXAi6qnKUH?|u6s9#rY|9KIiWzH_A-W@mFT_zvF<N{?M=(6;xn(m+R#?Ty#oaiF->ZH*c*xG@Wwu-7(CkEivi#ovP7#m?$#1}?2daPI4f*UQkUnP0gz#jY4u!87qbx4a7hQ>?Nt3pVFHcX0fQkAAedAef#K5mCw-5li@xTfg&t5*RxQA^;d<R-pSt=Rw%NCdD(8@%7Rn3`oZ3CCP}CQXp<DyBoJtGK1cloAa(dz2C00XPQ>Hl?eCP?BUlf_tKY<@Aw?Vw5OeQPB86Rea}-9`IQXa4Oa2Hv3M|B6gejlTFt3oF92Sjw9Wzw*%=3WCUb>!IcxdS`gvX-RL~Yem#OIh8<$#g@i<r%{1O6i-0V&V@J}4-6Cs12VeTmOW=sW!`H4mAnm#H(dL?dmE=3mD32=L0s_L}VvQk)*pDmKW;J0f0nizywX1;Orfk?~=|)}nLRJC*1-~75&tLrh<<r;UAKtxrefr|{b9C)ZF?=CcQG5XKSI@$yuU<`kIO2P@MjIt;a02FoFo5KqB&`-Y?9Js#EiNn&nZ=0^F*q#XiG1Z5trQUN1W7E6l5bzWeL?R*3NzooKYjk@yVJvOWJLB^^na*>g%z3lqE;gbj}$|2^3G%b@MZz3Qv=?E`W27O^;1!De0Vc^w1|<2z_{=}ZqPe}JU}UG{m5?)mtx?ckQ)i%gJKc_Up7t?(0$xkke~s*#@J&;L7t@%{_NZXTwc*PRAwK(8LS<EB%81((s>afGQw*e6Sx<E*f}|`p`?k9XGHRRrE@~UG5q;w)K)?Qlp4`s_N_@aYDbEbuu3^@#X5S<xbnFhDw?=8x5F6`Kn$@bFK&Mz?Vq=9i@RP&+PaH6cgJ0OX>6wUP4Zyel1dmD8ZtMM4+vNlqntD)&*c5&$KrnQ<NL4r(7J}HjW2CEf~(>c3ai9J6p^8Oww^{5E*cywPanvo0-7(sJ)N-Nmv9IC+<q%zGYr`sOVYKwUT1VZ3szRBhPaB`%E{*#TcOBMze=gg<ZbP+b!;|%q}@h-mcwD14f*X5WkSdrnNohB`_k(9pwbN0r6GrmbxxjA>Y*V$38$nkWjycTtu*~|R&?cx?4VwFenak75L!DpmX@w8t6EbAB>Sy@sPbNU6WGdoPIP#f*1rD*WlSBh`&mudqN?Q4!Y~1?*tcV5@0$_$9NEez>kJHKka0>9QdNEkQ%sT7o`mWU{YBH!Lwq{4zelGph-!O;cOxY8Sn9r4k7{J>X)>)^94Rz>qrNa|bGZ>X(S~x!2e1<{9S$;+aKG}E-(3_^5hDr@7x9N-vDyrG<e=GICyzIvet{~p-354`Tnx9#c3&Ok{V>Fg^3&s!hvUKT#^cJLty;zk$J<)MvO~4xxb5mU38QzQp^G8H4V^zY@%J|k6s!Q!Kg&K=L$eU7*@Su@%Du7d8Ddqk0aLljKtaIm3-lQ9V|bhURzdWsnMd1bmXHD0zsP|112zvKjz$-9Y_%K}fDQrwE}?D}?|<gMe-S==`s~{m;oGOD-)jFAM0+-v;}#uhEhdYC)kP4_7EfRP=>>2Rih2Q)=5r*gLhGquG$rkYp0>!zXp1_tz>M6)`?hf$TjYtu%PWZWBXWQiHc6mp@HEffy?A>1B7FTO{Nd&6=Wl+vZJ@$bqk>XwiodNIgk!e_Jeeu~9mwKL`LDx+>eD@<8s0DBjjW79H3#ASq}E`)gxua(Al4<Tj9)^c1n*EVF;)NHbIICb*{Qm^9VOXLnt^^5Dp4M|Mso@|y2isL_~shl1T1F9G9C?>)gc6nv|~SD$x*dSL1@1er9;zOgePH;`SzPWdGH|Zn`tuCX<2hUh5fuvE>>mzgu5?@rj_R3D_=^{m6N7fa)1<)=m@8dYIBSL(AHK|GgNj~!Fe5n=D=d|RdFA{`q(rF>Gbg<Cur^4GI&g8$0yX#P|y4Ek$U|Yy|bS1<cK4Bd_=E&m&1;_3>GPTtvUZ-woX@}8%05DqRh02K%;IR$-H;bZ!DzVaaCel=J3eTwr*v(J<x08)5p<sq`1aGX{J%9e%<TsfgX=tv$)GZ8(XkrsE#^ceB1}2ih|iA)MQ`?x#wbTl^|I=<N&1<TL?qyIn&!{<aYYDGAS{0%(_`)Go8#Q@yNEr>GT-iI1<#JxYEBi^S$Dg7_hH^Pgz6IMz*G)YT;5fEknb8mOk<nv!JfZD3BjVxtzeJX&DF0bsGL8@r+IpTTB0<6Tw<n1~kL}Bs!##IJW)P03-Qd(a0;Lqf6_ek$qiIYg(c1jCF?nxHDs&+~-)&MW+t9FLFlIa->c7q%yWuK^Bf$Pc>w1@>F}NsJJoS{C^=XcVO8Z)Q}$fccTxj3F2cFS9fs!ZUY&trQeg!@PKFr`L_bB&gk9@helQ|4MdMg%+(VdSEPq7TWfQeG+a~<3lM3=s5_`ZErRI`FtGzK9AFT|;mzfwcFisOYVakuhz6Hc<TLpQq3?*g@t0yk(LJ@*nB{brG-x7<N&;xNCep3Bq@^y_alFNxTLC99IRO??$JlgTs$N^OV#;9A090{nQzUHssk6PmiTmjt)Mj_T|Lsm$9UXNWi5(qf+>2xeR(GcIXs|5qY<x|s^#1_$RE`J',
 'pubmed_efetch_snapshot.py': 'c-pl*O>g8zcJ~4NAIf6{BsDEc8h^m6aXXM_<lR9$AJE9VYxFh<c6W6*EjHOE+0w{DAA%fm&M7}&54i?NF0l_eILQB)f0Fl6RjlGitH<kju_KABdiCnPSMR&{@ZrNZ@`sJgsw7Uu>&@haoQd<VWi?%j-=ANKo1|Kb*Dszw6W`}aR*E>AiP@%zC#e*Z7)MQ)a(cCfmsJ@)eE9H#2gz!k7nPWn*B?BX7x_xe;!0M@N(%Kx|2+`+?~i#V)j+w5lQi5^Nm}Z$>$oT-PKc^J(i6=ZiyxD<`6J2Auiuw>X8u^k)zbWtmujswJWZ2{UgxzvsHj)q<!HvHNMS-q80l$0Y;r(too>njzAx8lQho5i0tBQjtI~S(c9n**Dx?h0Q@H}VU&8MqF2(sJj+)6i4v8xHw(_%lCWF%t9tijkNc*~wr7W(cSS4?POG%a{nG`sZ*pzQ75zl5xwh-kyo=Tu$RwgqkZt`MQJ^^H^ILX8$&8Jsll5etETqII44VOsf0#_7CDP96o)WY~5Ox+Zj@H~D*AWAV5ci!3?2oFdS<Ih=gjX%<a|1GP-q>?NAV^Vlqo{Jm2iPI}z^7aF3nROx&?V%W{mmzH~FJR?yJ!}pSrpqKn5(qWYf?_ZPKr}3@;zd{xZ+veH=$r(j!;^8C<~OqN1EGmhx-fto^tb`KMl_7qYnjb_?=RV3GB0q4xkG5jgiH|p|E;2To1{#$po4=Uo@1ZqsrULFEx_+Rp$EoA6*B39fGCeokYuzal2G;DL-cSzJf|v>wU4`Hp$0}4@wAExIR$a_`G;mcOyUQ~Dlr@i4>ZwA&d!P|nWoaSh%-&9xJvR2n5(AD6QmC4Ej$iS)oXk2F+)LdP;b>74W6(zfI<ATyrLrj(gkPzC21%_@>Md6=$y3Rp`Ls{U{zA3QbSXCy2xGP2nMS6aSzORQWB}@AdMRn>q#R&gM)dd27Aoh3^{A#5vQqd4YE#Q&n_EsY>){-08mD0)mPP07F|m_1f{hbLls+b2i8ine7Zp`&=iWV^8D(VdS^?7%|ku6#X|%1eR$Y_)@I_qP}^LzXYRvB2bG<C?8519<70d6ZayL-V1vfu8O8*bNkTgs&NpegLWjdIywS<w@5ZBJ_~!t$InJ@IND(JMmp{hoMxGZ%UVs?9`{}QlSS%Cq=NI3IYMH$I>AyGZ)hyrr{MWTe-~Ig0T0V%KHcmBv)re9~QAWY!qsCzd4dDO$7HOcVH2^)FED}Cdnt~RpX`C%K@d7-dE1(mq05vFFNKmNXsMlygy<pIIdwIK7Z)$*xBHyf|8ZFmovURUfX-mc>Y}gB$#x$SZ+61vkQEy}-Qy7D<H16<((bKlToiYVyZjl$a&a9W|>Zy9ZYwj!olM~-YdRhJnJM;}yR8RW%55b^(`||9~o9DlOdH&4fgX1-Asb;ITrgt7Q&pTCIM5jh`<kaGwS0N`(iHC!>8F&8k2vwR6KVfc`t|E;nJl4M=ROH1eQGeUvG{EC7pP`WR<Si;9n?&IK5E;EwIHZpRK=tU{J7iQ;X~VRHO4CLy^N7$09A#pSK5b#GjsseFS0Qw4#aWPt8_D%xX@Bks|4y@E8_<^YpN7Y@-6pFUw9;f2!fKdag~T0bX?sTYxQ^H49ocTuh}6fxg*n!4k^;1h%SaDEEs)*%N*YHj24jM$o*`H*tXd_3+1%if_p&^yiKNu01D*)ud#j`@AykM6Ju{M(AJZ<LPUX6iGtYGs!RH5s3nGL+P)rHHsZjg$%tkee@UGUFTB`9!<E(hR2^>zpz+fA3cy9j*ZzXDXlx2)9*vq8gERu7KUx635X&wWPDh!je2yreguXEyk@!|Q)%jcKhc}P_?Biw+{L1eJgsGupC{_-D!r;DrQFaHUV*Z=&>|Nc9elqnjPe-%{$zZ?4HIWqVBGQ@Ba4`jB`IBK^BL6hdLm91q}5kd$d6$LH8FbZrGu5_l(FD_nP=uinpFua}HP_aJEM703N<%xFwd3sRWVnP62^beaZ`Wxw-;nq5Hq8E|CQ0kYTvc;*m(?GYipec3%Axf|cP4?+g=;(oz%r+2ypx)%yaMZGV6{iWAz`N~$U<9s#R#BrS>Zzl_Oz;UC#Y2&x(7OT2KyM#CO#N`%^qDjswM9V_ffBpP<l8kI3fL2JTCIp+VNn=hq0rR*(;RX&*~WH2i>4*Oup2hth`DscU5}M-@rYY|!bAi-+<zlUb`5W4Y6;?iXB|zDav$IY_isXZf{6KuHH_<{(tE`XBM`re;2K9ujopu)#(-8Q!E3DDkirl$G33n#d!txGh}X$|ST+J6>p+Zf=va}mmUxYQ15YQ(<Ie9>^q$wF<<Se)`W*)C@PwL*{p%REGq5Z`Coy&bG`4^|`~n<)k%6E8^FMCH#CZSHU;Yae5BmL2#Pr=y|CWKv_U`9@TET=ERMt(XlT^yo85Lu=hIZzI2Y)=jc=P<#OEkA@(gKHUaSpFfz<oM@`Rw(p=Pxhu^|Gqg<>}Fpq@-q;O(#hR9$%Ox%W#ojAHi`f<d5q05km>*(?tSso}GVv_U$*9Z-^C?YmLg}N94fC@i7n!(t202Kx&=9UyFE+yvVZ|J_$cX9#tR#>k9w=xM^!KuL%i5<l4z`I1ry5hseFjX0||0E>1o_4uKdi&i))-oWFVf>gAjB=&SE8&k>uG&%^=X@c0w;gNcIXmvjL#4e*#J3!i^zJIa6Ib`B2o<_6;6fS+%Tl|gq}O&!@A9|t->g=Dj!c%~93;)|i^8jR0@Q2Q7^Omx>%#cESlVggB7G`ir9gWm}U=OhdPT<8VecxrC6kZcV=<9V_VTANa$HijdT-U<x(8ALA%hTpw7)FZ?f`{AHx)I<^MyiBU(T594Ld{E6?NeEq5NtUcOD{_B*M=}Q>nP7^2Eokuf@kuat?Jpq?&M9bvY-+X#kU>IySacxf;4@W_k5n>%8U8RSA({o9XO9o8&Ewy&YlT}=ceeeN=WN+@G2{;BZ^JrCK*vYaqNfj3?R=#&^VQP;dHxuYLD#;qzw5nrbxqH&QL2xL($`2{4Sye2YMe7EXQilaa>1Z$hGv`91Vl7!c>KJ@`jk_W27TBFN*+G^UoX$a8X$@;h0sfWN4Xoy&pD}_Bi~Bq1V?|5`5b~0Vu7X>boFqVmsJ)+Dd|I_ds|A?vj{j{?xc*WcYS;D4aOEQi8UCRh$a;XT1dg7e-hNxD4;wwt^8{Qs|le3L-c<5d3`_#d*B@8I-Y*@JUV;*Jo<<8@6e>u-{&vRo`2(HVBqKg1^_LGh&8)T3doLPt^@%!vOhp49^H^`g%17*3ke}VNF5IZ03+UjoQ)ydqTkkaEj2J`jC4qYfJdh%pB^8NZMuM88pU%6SM>335Xo^{p|-V25lcD+4%r+gy!Fvs@o0*I++(6&PzZY*s<4XPT)$@hKw$Qaezj@c_M{u@`knPig{?R=rOhzQZ+u<dgo@@g2*N6_;?$5LXuILv_zlFXYgqtQZd&MC2;x@pf73KBOQo}(g2!SiW5|*yLLx;G@HCwd#1vefRKfzF6$a{*W3&W`fjk>x64;f)bRL>1^cT(J=i0QJ0a#G=Q8f*W{%|0vWhlhs-$;7Zug%=JE~JKUc!Zr0w84c-8P8CKZPv&N@UFjwlIh{u0;4PN4$rbF(&SGNdY6ZM`VW~{M<?Mip29QAdf-@a(8&b_utyN2dE4fi3-Y32pR@_4hd9!DDN;B`I4s19wAYvqTZ|?Wlme!keMPwt(*Y3<_uWMT&XCrV0Ndj**qLu6lG;EZ=LKb1kUo7tNReszn5KCtY1tOzHGO%AGwUW!FvN^!w@R*N@H?i-Y#=vKnv`UMLOL-4nBoQzT7nJT02x-ZhXx{?Faoj4VOpLgQ@@=Ut>>Ys?8qy!8MtI?=$c3<o5RGT3shZ)XQ;a9Y@)s7W(UPn!S2fw(nFxnQ=o6{2i1ikK($PzT>Id4_$T3@#rH#-B!e3Fz@adwvzWNbPmNdWRF+Qh9K$)xg#&~AS%TSpFwx+(l)h`Q%aI2^&`G46B9*@#pmw1QD*G$0^v@=EC|I`fpn&38LEau8;^}}V6fAlJoe8B}4Mv1{o;J@OAHAn6@;d6WP2_-=mK8hSBi}ZUm|(zG5szHSrX9tA5w`6)Zl%sYw{2+9l|`p*j|>W9$NY|Pnw6a}(m8BNC#2IX{W#KY@7Oju*s1}gq`0h9CF!m~59NbX?g7%<(mxrvP%XFq5i~cPHtQNN)H<Z*c)G*nH5X9tJlOIwCtgBegcL<_ITbptts_f#sD4w4ic)0@%Dz|~Bm4rBFmXj`;o792k;EfFA#-FB0cZlYjr}l)4r(`gBXs(A9d9tZKn@8uE&=s6pyf-H>xoBd6zpxWfftmBXq@7rtIpD+ofK~E@%t#Idz2GvAeJCdOH0@lEY;FFZg^5b9z~M+xwh3y9YHvPWP+Fum>&tNQ49p*3A)S6WHyr-krwl<02F-01o9>jF#;fJKZ3%IQmg_>Gf<mv!=^4X$IjsCtObtG<B)$0^!A48=Y2J|I{B7@7Zn){hbrv^Zn}(DLzI936QH2f*O1YYJuEV{7CkQHCy?}*$?hq1wzN-T3HVS~Ixyh~Ln4glhX$^8X2|eA<spZ4*|*2`leiOlZ3ny?v}Nylp-z{Z><TcY3<>M0<L;5%o&}>hS*jMqvPGn}>3^nEN5N@7?`Wk*LyDsb-VR>}9UUD0rD$iYPIrrkQ46rEj^p2bKs&-~y;I{cK$`4D@)i0Bv&~dm38u~?Ywe_EM7}gC=5!zAI<-dwJ<y-FKG}&9v%YRJ>t}^hxvW>&4dBT^pSHV+%4MwmXKe~*$pW0LMs2KXu5!NT9{i!Qr$=`R=;-wEcx(RJhKnD)p7U%k9as4(nMRY_N|wIm)a%s-)N=|7a|I>vvaD;2E9BbnxbOHl8bAXZ)=(9ik3aovsD9;Vgvbz&fTl*z0(b&gF=49d_;`n;*b=YiV2SIK<Ie};lgGash)*%U{OR#y{O4o*=acOYXzkS5A1*Ini#rCreF6lQqKvQW#$uxs`lPO%cn4IwYrA$yu*F$p=25pd+#MBjo5=Ele>tb;V1fq14#%Nt`x@8R(YbZRg6^W3;rd7wC&r|cDOoi4?kIDMrLi!I(4UN=E!0Bc!L~^hh3Qu@tgexJk0sj+^@K$3W5}m~xxC%6LK?3j6=G7mv_0%9_FMYP_(cQpanK78GdSgveZz56C6IU6p4yJ1?GW=3<!u}f$4*9%m221sSFe3jfbr@I$-z~(<MGobI@_Sf)$#k&1tp~lAhZc{P`AE^jny&(*8=>fp|`2#ho5^SaqPA9`Y8C4G4~N*INPk&rB9?rfaRuu(pou9683_qPo~{pTt_Hc{S$<aQ7V3+*R8Ni9mGyQS~ZnGjtd<>Afl?SVL6ZkQfz{<1Jr$k4U9LmhNX_H5~R#Lz;aiub)EO>(Re3d#V+h+Q6ZdXv=s(M`uB$FTtnMOWwYc?3jk}js4KQ|V<JY>em1_>1vUjzoXlPyKM52y!8hAbj*M&#Db;PKpkSS1Q!d$EZgHBsHMM=Qg4f}&FJ|qKiU~;XRig<po?hkiIf&R^XL<nTmV*P~>*Wt2E)F{4Vn=rv$o11fL!l@%x<LKr>OyDl`tn8M%bl6AKDuHkXReZ9a7Mpjoh(mh`Azl&YJ8N?(%CIE*A&qu&>)uQR{REOmLA{D67a^^E#Vdn=4VQlppcZCbn@Xl$?0lM20n*m5Xfoi!5grxRH~zl>;0kO-QHTJ>&)LDdG;PWfM?txPAcLqu-%R`E@g!J7F+b<N^cOrAkl0=VfX6-&MvUOD`CJ;pJ0)zR2cxFAcqjrLu{_?&vhbiLb$d?m%KWyl1X@0IiGU!;OdG`tQ&!Rx3<6}t;>YuwVMioV+6_ZBR)R7<7p#IERLC>51qDfC!^iIu2eVFtA40<irp<T`Z$hxln@{4zcB?CP#M>^vC>=9Db;=Iu=RyG@%-5l)*hi3ldFr3x*B)zX!5S&E7>A1FVC2&<gg(#cT_c!zYe-8#hgzm@8>l5)O|dLIc__U^)LeUk<1f0Gp#(%x$smU8aL>;)(*ShplfDVM^Tw?im5d1LeX>?Q`gIJ-7$i@cdvu!P>c37<gPN_AArQY>1WhkYqjU<)z$kpRz0@1iyR7uHqunI-n+4>bK65WeLh;~QloBV@p~~2oR&#y4u>E_i_XCT(2M$T)baRX1CC1tiQ;@3pbG#U+j8hJ!>~TDjw&2o>v7i})JWW@%6BygqFARIj&)0NKT#AEEl?nE1_bJ?{m89WQ^+}sn(mVtDK*S(o9o8TPV7C#5ZcM=HS3Acf`%NG)sUTfteZjgw!pK|@b^1cqW}%h^w2reAj+^&(C?&ShIJ0rBppaeIF4`^cq<`cO)#FZs1y0-y|#|f%`qO?nf`I#26VjF0Rk>)HbTNqAdM(=$Pp->lFkqn&q`$~UiN}$h45gh7uqQp2N<(<FU=Z^ord2IkKXn4nBnN|3LFr!OILjSuehT`6>e0;#;VgaLI|v%HBd4=yTNw1Pt%Y2*uAqwX9xy2k31r3Us4!!9#Y$q&QrV;YK|iIsIw*;c`XfNm)7(krOzUS0|Rs<i+=|FqYcJGbo|ga+No@Y&AS(@J_`9gbsc-{h3XVd?Pe$*G!)~7X^5vgYf5Dax|QxUsX_B==ykNP)=mT}p<9;OhL1+iv_nxs;xR%qpJ&SDaQZ>;<^;DxA<{Z@C-+E2@B27H?4MFv4~kI?1eGF`IKf}tP}|2ehZnbWX$>0JZbY%kYA-3i!Nv*Ok8J5SY-uX<^ji9j8->vAo0PCOh;*R!4Z3-J2M~hrtjT+Mc8Cx;x3ZkZYe-rGw)Bt-PPy`z^)X^nTW85A9k(CSVF{7kc0q~i{NTQ(MwjqvdG<k%)&)>H!-ho70)y*E;;i}2hZrSm9VdLxDZxykz(5Vm9RwG}AxNq9U5NVg1VDPc#s%d=6-BDMg_KkW`=`_D$Q}wOI1z7ML21O9bpH$~&|#$ZWf`>oExUYhm4l9&xo2E+pcmvAas0k)rKp9=$M=R!?&TCcvHg_>$Hm(jEvUVbEbWR90v%E`u1B;(^IsLrX+-w9*Fe`<c<T;5jDl*+GS7XphSq^lk6`3bwRDmkxV}c6pJQ#dTLr>bGcF|RDDCW9IvA#;w@)ayf;tO&Qv`X-uoaRZtdzsoEB<f^LsbiEk>6Nf;;<H{Dn3zHKMbtqI(3L{Bi4nwbHo)xwI_=}xO$J^)k}9tA5KK$2R~W&cSeDMJEmb#`Kp?^UD&*AOnZSofC^X@LocTfaNM-*2(-Gg;SP?tYpYI&ow{@i86w(KH%+h?6$lJCv`$kL<u43Ftny&X+%&TlO`BuE*ky9Mb=ZYAk1fN)!J}g>cW7I}tV7-2qjC*nIu!0b2;ZxHsqhdN3+jBHKeX$+tz}iExAh!rz7MT0hjf*AIh)8UHEX-9=Q|3!6V=OpU)6$Sr^}Z&nZ;kztVd9T&KSWicj!I2MUEDc+_WaYHZFamrZF0KFUnk%jJCS<XS9_SPkba^!I4Ay;zeXmp@N~h)m4L7-Dm(s0g2-$cna$*H6Ky0v_^nMz`z00I=5kiJN7QHU4=3pl@qqoUY#h_2}$X-6MzPF@YA5vxS0~~&xW&1?2`rWu(BPw@2;eG%V^*0_1~3jn1Oe-peaG#S|-r4q6DYUdg^BKg8>#*BCvvFJO{?3k61&gr>=>GX2^UINgOu(`1n}sLCX}@rq48<_gp<QexpLeX)Q-9<x_oJy=~r`(v!Y$yDHm?O%Lq9C57N{QKR9Ym_d;tN@ttsw%)}{FP0)0?zs9gHVy&he!JPK7T!AFUb2|t^nUMfl@y#7Q(bv67f{hY;qo~iIKM}hq02qZC%$*UqU;#Kektxr><>gUiq)T;;rd6D^qM2d&RRyGUGMQO;j%lk&1&km(W#oCAWZ3^27VjBy0_tXUC%I5k3z!2bKG^ChW1pcPf&2vFytetw96=%YZ;9jmJJimPE?IJ!8~Qik2)cYn`K)zmAu7}11Pp&?!ezpZ~n%*b1(hbBWLw}a?LN*&)7u)Jn?4S-_<%0v`;;SFEkqN4;`L%eh;SAZomnBEXA0VzGpy2@O7jTt_fY|oT<i7(&%@4_1StQ&~>&+J3+YusN}}@LUs042i4oIZi9Abk`gX+H^ubY5ceTb0YX#uXs_l~N43}5&g}%^Uc>0LP`WnV{w{_6K?<!M{hj6QrgpSXJde4r%Gr(%)x_>Jg}s519l@hhrC#i}iQU*6HE|#A_4gCh*L%pZi}k%2DsZT9rjD|Z86kXe0B!PH{T;*DWN0`_tsPy8FwI9>ZtO2<-4|G!T*dn(b=0)(lnxj0YkHTh6d+z1X?>8ZtaR}nV^s#iJ2ieQX_L|2c1+*H?8&+~wL&Z0AMQ#V4LWLno;vxvn=&W^KD5y_eM-(@7*3#FMQsb-xjBYC*Rl(CHo<sZyJ8Yp^|SU!6Lr~Uh%!3*BOg$(Q6DC3S8WozVq=%u>p!jf20U!!Q}(|8Q0r*7B4WPnq09d5a*^4gsS_FXty@#fH(nqGl-4)VkPMhU;m-%bVdAeA{_VTgh>YF4)!YhEs!L-g6N$HiaT20Hx5hfkzjX2oP9yP`fo{kTFFNZ?<cB-e<8iQq6=p@Z_Vu3Y*^~EMPL-~DS8A2JKD3i_W9JgxWHX;bBm)JjJbXjnuX_H<2RAmQ1_7^Wd2uAP6NS$}gZzme_zYpdGJ@<Xhb%6wx$L^}H5I?-EpCH)cFJI{G!J-sxaJm{oY2!FWbj1N{cYcPlU#N`ER%&f(HbhXK{f0fa;dgU+`emBdwndhS~tzpRJCR3Pe?z-zDrr~AaMHds^PB2VCNfICAV$XVE}%2B42bhMN#81zN5b4{acD#D`8Rlgk*^t577?NJX#x`SfBpg%N193bO5`Gp+ctI@VgodPy1@hNmOUJc3Lsz1W5Iq8~Rj)G6(Jvun&-xkk-K3m0<g5>;T0RC}l?0Q?a-YK^Kq%31rjoUv~x*iY`~wtyeo&hlHv{a2q`MKc@e-r~'}

EMBEDDED_SHA256 = {'audit_snapshot.py': '862203e50f60f4f3121926a44b7167decde38a7d2317dd21a5e7065a0162aed1',
 'corpus_runner_snapshot.py': 'b0261d274f3fdf99731f4d3b34adb503418a7563a0327b7396e0e8a34eaf9984',
 'crawler_snapshot.py': '47cdf3f94b1196fb26e03d69bfa7bd830c64a26a72ddf8f538df28f327d4694f',
 'diagnostics_snapshot.py': '30f58690bee266f40601f2967b20c849281b3ad22e0977b70a0c765e3f84cb7b',
 'export_snapshot.py': 'ac461b827b046ed7ea8fc4488a85d62392e4a95920ca9be7e8d714d3737a4023',
 'postprocess_snapshot.py': '377d747c79f82e52d4fd9f16e36512af540d985536aa0a70b86855eff76842b8',
 'priority_probe_snapshot.py': '1021c7ffbd10629fb3137d0102c8bb43106928d785bca3293125f9a199b55604',
 'pubmed_efetch_snapshot.py': '6d107894f08733407536fc1e99302a7882a1db5af5a505b7d50c93555c0a8bd0'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
POSTPROCESS_FILES = {'audit_snapshot.py', 'export_snapshot.py', 'diagnostics_snapshot.py', 'postprocess_snapshot.py'}
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if name in POSTPROCESS_FILES or (not target.exists() and not POSTPROCESS_ONLY):
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
import importlib.util
postprocess_spec = importlib.util.spec_from_file_location('notebook_postprocess', RUN_DIR / 'postprocess_snapshot.py')
POSTPROCESS = importlib.util.module_from_spec(postprocess_spec)
postprocess_spec.loader.exec_module(POSTPROCESS)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

def save_checkpoint_archive(failed_step=None, error=None):
    archive = POSTPROCESS.package_run(
        RUN_DIR, WORKDIR, id_from=ID_FROM, id_to=ID_TO, dataset_revision=DATASET_REVISION,
        source_sha256=SOURCE_SHA256, packages=versions, refresh_diagnostics=refresh_diagnostics,
        failed_step=failed_step, error=error)
    print('ZIP checkpoint đầy đủ:', archive, flush=True)
    print('ZIP chẩn đoán:', diagnostics_archive, flush=True)
    return archive

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException as exc:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        try:
            save_checkpoint_archive(failed_step=name, error=f'{type(exc).__name__}: {exc}')
        except Exception as rescue_error:
            print(f'Đóng gói cứu dữ liệu lỗi: {rescue_error}; checkpoint vẫn ở {RUN_DIR}', flush=True)
            with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
                stream.write('Đóng gói cứu dữ liệu: ' + traceback.format_exc() + '\n')
        raise
    else:
        refresh_diagnostics()
with diagnostic_step('snapshot_validation'):
    if (RUN_DIR / 'crawl_config.json').exists():
        saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
        if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
            raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
    if not requirements_path.exists():
        requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if sha256_file(supplied) != SOURCE_SHA256:
            raise ValueError(f'Parquet Input không khớp {SOURCE_FILENAME} của revision đã chốt; cache hiện tại được giữ nguyên.')
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if not POSTPROCESS_ONLY and Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

PubMed mặc định dùng **EFetch GET**, **100 PMID/batch**, **1 worker**, nghỉ ít nhất **0,5 giây giữa request**.
`PUBMED_USE_EFETCH = True` hoạt động cả khi `BROWSER_TEST_MODE = 'off'`; run chỉ có PubMed không khởi động Chrome.
Giữ ID/URL gốc, ghép theo PMID trong XML; `text` có title và toàn bộ abstract cùng nhãn.
Các domain khác dùng Requests rồi Selenium theo cấu hình browser; `RESPECT_ROBOTS` áp dụng lượt crawl website.

Checkpoint XML và batch được lưu trước các record ID. Resume sửa các record còn thiếu mà không GET lại batch đã lưu.
`MAX_NEW` tính số URL corpus mới, gồm cả PubMed. API lỗi sau retry giữ batch pending và dừng để resume;
PMID không được API trả về hoặc không có abstract được ghi riêng, không đưa vào corpus accepted.
Xem `pubmed_results.csv`, `pubmed_missing_pmids.csv`, `pubmed_no_abstract.csv`, `pubmed_review.csv`,
`pubmed_requests.jsonl`, `pubmed_efetch_summary.json` và `raw/efetch/` trong ZIP đầy đủ.

Run mới có hậu tố `_efetch`. Nâng cấp run Selenium cũ bằng `RUN_TAG` mới và `RECOVERY_RUN_PATH`:
reuse raw hợp lệ của domain khác; PubMed HTML cũ được lấy mới qua EFetch.
Muốn thử lại PMID thiếu/no abstract cần run mới; resume giữ những kết quả XML hợp lệ đã hoàn tất.
`POSTPROCESS_ONLY = True` vẫn audit/export run cũ mà không crawl lại.

Tham khảo [NCBI EFetch](https://www.ncbi.nlm.nih.gov/books/NBK25499/).


In [ ]:
with diagnostic_step('crawl'):
    if POSTPROCESS_ONLY:
        print('POSTPROCESS_ONLY: bỏ qua Requests/Selenium; giữ nguyên cấu hình và snapshot crawl.')
    else:
        # Pass credentials to the subprocess through its environment; settings/log contain none.
        for name, value in [('NCBI_EMAIL', NCBI_EMAIL), ('NCBI_API_KEY', NCBI_API_KEY)]:
            if value:
                os.environ[name] = value
        POSTPROCESS.invalidate(RUN_DIR)
        corpus_settings = {
            'user_agent': USER_AGENT, 'delay': DELAY, 'attempts': ATTEMPTS,
            'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
            'respect_robots': RESPECT_ROBOTS,
            'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                               'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                               'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                               'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                               'robots_failure_ttl': ROBOTS_FAILURE_TTL},
            'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
            'pubmed_selenium_first': PUBMED_SELENIUM_FIRST,
            'pubmed_use_efetch': PUBMED_USE_EFETCH,
            'pubmed_efetch': {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
                               'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
                               'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT],
                               'budget': PUBMED_EFETCH_BUDGET},
            'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
            'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
            'browser_attempts': BROWSER_ATTEMPTS,
        }
        settings_path = RUN_DIR / 'corpus_settings.json'
        settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
        arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
        if '--log-every' in CORPUS_RUNNER.read_text(encoding='utf-8'):
            arguments += ['--log-every', LOG_EVERY]
        if RECOVERY_RUN_PATH:
            arguments += ['--recovery-source', RECOVERY_RUN_PATH]
        if MAX_NEW is not None:
            arguments += ['--max-new', MAX_NEW]
        print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
        run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Audit chỉ chạy khi đủ checkpoint; raw rỗng hoặc HTML không parse được được ghi cờ chẩn đoán.
Sai hash, ID/URL hoặc thiếu raw của bản ghi extracted vẫn làm bước này thất bại.
Corpus chỉ xuất sau audit đạt. Thành công được ghi riêng trong postprocess_status.json;
run hết pending chưa đủ điều kiện thành công/upload. Nếu lỗi, notebook tạo ZIP checkpoint
đầy đủ và diagnostics trước khi báo lại lỗi gốc. Accepted là QC tự động, vẫn cần review nội dung.


In [ ]:
with diagnostic_step('report_audit_export'):
    summary = POSTPROCESS.run_postprocess(RUN_DIR, SOURCE_PATH, run_script)
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending']:
        print('Run chưa hoàn tất; chưa audit/export hoặc đánh dấu thành công. Chạy tiếp ô crawl.')
    elif summary['accepted'] > 0:
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Audit đạt; không có tài liệu accepted để xuất corpus. Xem QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
with diagnostic_step('diagnostics_archive'):
    from IPython.display import FileLink
    import zipfile

    refresh_diagnostics(strict=True)
    with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
        diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
    display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
    print('ZIP chẩn đoán trong Output:', diagnostics_archive)
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP được tạo cả khi xử lý thất bại; manifest ghi complete = False cho run chưa qua audit/export.
ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
with diagnostic_step('checkpoint_archive'):
    from IPython.display import FileLink

    archive = save_checkpoint_archive()
    notebook_manifest = json.loads((RUN_DIR / 'notebook_manifest.json').read_text(encoding='utf-8'))
    print('Run đã qua audit/export:', notebook_manifest['complete'])
    display(FileLink(str(Path(archive).relative_to(WORKDIR))))
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
    if notebook_manifest['complete'] and notebook_manifest['accepted'] > 0:
        display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
        display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))


## 10. Tự upload output lên Hugging Face Dataset

Repo đích: **AncientWings/AncientWings_R2AI_STAGE3_ds**. Bật Internet và gắn Kaggle Secret
**HF_TOKEN** có quyền ghi vào repo. Run All sẽ tự chạy ô này sau khi xuất corpus/đóng ZIP.
Không cần đăng nhập qua widget hoặc tải output về máy.

Mỗi run được phân biệt bằng `RUN_DIR.name`: nguồn mới mặc định `ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh`.
Recovery có tên riêng; nếu crawl lại cùng khoảng ID với code/cấu hình khác, đặt **RUN_TAG mới**
ở mục 1. Chạy lại ô upload cùng run sẽ cập nhật đúng các đường dẫn của run đó.

```text
corpus/<run>.parquet                       # Đầu vào RAG: chỉ tài liệu accepted
corpus_jsonl/<run>.jsonl                   # Cùng corpus, định dạng JSONL
archives/<run>.zip                         # Chỉ khi bật HF_UPLOAD_ARCHIVE = True
diagnostics/<run>_diagnostics.zip           # Gói chẩn đoán
manifests/<run>/notebook_manifest.json      # Khoảng ID và số lượng bản ghi
manifests/<run>/upload_manifest.json        # Danh sách file, kích thước và SHA-256
```

Mặc định upload corpus, diagnostics và manifest; không upload ZIP raw/checkpoint (`HF_UPLOAD_ARCHIVE = False`).
Mục 9 vẫn tạo ZIP đầy đủ trong Kaggle Output để tải về/resume; chỉ bật upload archive khi chủ động cần.
Đặt `HF_UPLOAD_ENABLED = False` nếu chỉ muốn giữ output Kaggle.
Run còn pending (ví dụ thử `MAX_NEW = 10`) sẽ **chưa upload**; chạy tiếp mục 6 rồi chạy lại mục 7–10.
Run hoàn tất nhưng không có accepted vẫn upload manifest/ZIP; không xuất corpus cũ còn sót lại.
Upload đọc file từ đĩa, không nạp cả ZIP vào RAM. Nếu mạng lỗi, ô thử lại tối đa 3 lần;
có thể chạy lại riêng mục 10 mà không crawl lại. Kết quả commit được lưu tại
`hf_upload_result.json` trong run. Repo đích phải tồn tại; token/repo lỗi sẽ được báo ở ô này.

Tài liệu: [Hugging Face upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
và [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).


In [ ]:
# ===== UPLOAD OUTPUT CỦA RUN NÀY LÊN HUGGING FACE =====
HF_UPLOAD_ENABLED = True
HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_SECRET_NAME = 'HF_TOKEN'  # Tạo trong Kaggle: Add-ons > Secrets; không dán token vào code.
HF_UPLOAD_JSONL = True
HF_UPLOAD_ARCHIVE = False  # ZIP raw/checkpoint chỉ giữ trong Kaggle Output.
HF_UPLOAD_DIAGNOSTICS = True
HF_UPLOAD_ATTEMPTS = 3


def upload_current_run_to_hf():
    import hashlib
    import importlib.metadata
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import re
    import subprocess
    import sys
    import time
    import traceback
    from urllib.parse import quote

    if not HF_UPLOAD_ENABLED:
        print('Đã tắt upload Hugging Face; output vẫn được lưu trong Kaggle Output.')
        return {'status': 'disabled'}

    run_dir = Path(RUN_DIR)
    manifest_path = run_dir / 'notebook_manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Chạy mục 7–9 để xuất corpus và đóng ZIP trước khi upload.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('id_from_inclusive'), manifest.get('id_to_inclusive')) != (ID_FROM, ID_TO):
        raise RuntimeError('Manifest không khớp khoảng ID hiện tại; chạy lại mục 7–9.')
    if manifest.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Manifest không khớp revision nguồn; chạy lại mục 7–9.')
    if manifest.get('pending', 1) != 0:
        print(f"Chưa upload: run còn {manifest.get('pending', '?')} URL pending. "
              'Chạy tiếp mục 6, rồi chạy lại mục 7–10.')
        return {'status': 'incomplete', 'run_name': run_dir.name}
    # Completion requires fresh audit/export evidence, even if a legacy manifest says complete.
    import importlib.util
    helper_path = run_dir / 'postprocess_snapshot.py'
    if not helper_path.is_file():
        raise RuntimeError('Thiếu bằng chứng audit/export mới; chạy lại mục 7–9 với POSTPROCESS_ONLY = True.')
    helper_spec = importlib.util.spec_from_file_location('hf_postprocess_gate', helper_path)
    helper = importlib.util.module_from_spec(helper_spec)
    helper_spec.loader.exec_module(helper)
    helper.require_ready(run_dir, manifest)
    if type(HF_UPLOAD_ATTEMPTS) is not int or HF_UPLOAD_ATTEMPTS < 1:
        raise ValueError('HF_UPLOAD_ATTEMPTS phải là số nguyên dương.')

    run_name = run_dir.name
    files = [(manifest_path, f'manifests/{run_name}/notebook_manifest.json')]
    if manifest.get('accepted', 0) > 0:
        files.append((run_dir / 'corpus.parquet', f'corpus/{run_name}.parquet'))
        if HF_UPLOAD_JSONL:
            files.append((run_dir / 'corpus.jsonl', f'corpus_jsonl/{run_name}.jsonl'))
    else:
        print('Run không có tài liệu accepted; chỉ upload manifest và các ZIP đã chọn.')
    if HF_UPLOAD_ARCHIVE:
        files.append((Path(WORKDIR) / f'{run_name}.zip', f'archives/{run_name}.zip'))
    if HF_UPLOAD_DIAGNOSTICS:
        files.append((Path(WORKDIR) / f'{run_name}_diagnostics.zip',
                      f'diagnostics/{run_name}_diagnostics.zip'))
    for local_path, _ in files:
        if not local_path.is_file() or local_path.stat().st_size == 0:
            raise RuntimeError(f'Thiếu output hoặc file rỗng: {local_path.name}. Chạy lại mục 7–9.')

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Tạo và bật Kaggle Secret {HF_SECRET_NAME!r} cho notebook này; '
                           'token cần quyền ghi vào dataset đích.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME!r} đang rỗng.')
    token = token.strip()

    try:
        from huggingface_hub import HfApi, CommitOperationAdd
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'huggingface_hub>=0.34,<3'], check=True)
        from huggingface_hub import HfApi, CommitOperationAdd

    def report_upload_error(error, stage, attempt=None):
        # Giữ thông tin chẩn đoán, nhưng che token/header xác thực và query của URL ký.
        def redact(value):
            value = str(value).replace(token, '[REDACTED_TOKEN]')
            value = re.sub(r'\bhf_[A-Za-z0-9]{8,}\b', '[REDACTED_TOKEN]', value)
            value = re.sub(r'(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+',
                           'Bearer [REDACTED_TOKEN]', value)
            value = re.sub(r'(https?://)[^/\s@]+@', r'\1[REDACTED_AUTH]@', value)
            value = re.sub(r'''(https?://[^\s?<>"'\)\]]+)\?[^\s<>"'\)\]]+''',
                           r'\1?[REDACTED_QUERY]', value)
            value = re.sub(r'''(?i)(\bauthorization\b["']?\s*[:=]\s*["']?)'''
                           r'''(?:(?:Bearer|Basic)\s+)?[^\s"',;}]+''',
                           r'\1[REDACTED_AUTH]', value)
            value = re.sub(r'''(?i)(\b(?:access[_-]?token|token|api[_-]?key)\b'''
                           r'''["']?\s*[:=]\s*["']?)[^\s"',;}&]+''',
                           r'\1[REDACTED_TOKEN]', value)
            return value

        chain, seen, current, status = [], set(), error, None
        while current is not None and id(current) not in seen and len(chain) < 8:
            seen.add(id(current))
            message = redact(str(current))
            chain.append(f'{type(current).__name__}: {message}')
            code = getattr(getattr(current, 'response', None), 'status_code', None)
            if status is None and isinstance(code, int) and 100 <= code <= 599:
                status = code
            if status is None:
                # hf_xet có thể bọc HTTP error trong RuntimeError mà không có response.
                match = re.search(r'\b(?:HTTP(?:\s+(?:status|error|code|client|server)){0,4}'
                                  r'|status(?:\s+code)?)\s*[:=]?\s*\(?([1-5]\d{2})\b',
                                  message, re.IGNORECASE)
                if match:
                    status = int(match.group(1))
            current = current.__cause__ if current.__cause__ is not None else current.__context__
        versions = {}
        for package in ('huggingface-hub', 'hf-xet'):
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = 'not installed / unavailable'
        detail = '\nCaused by: '.join(chain)
        heading = (f'Hugging Face error | stage={stage} | '
                   f'attempt={attempt or "-"}/{HF_UPLOAD_ATTEMPTS} | HTTP {status or "?"}')
        print(heading)
        print(detail)
        print('SDK:', json.dumps(versions, ensure_ascii=False))
        diagnostic = {
            'time_utc': datetime.now(timezone.utc).isoformat(), 'stage': stage,
            'attempt': attempt, 'http_status': status, 'repo_id': HF_REPO_ID,
            'run_name': run_name, 'sdk_versions': versions, 'error_chain': chain,
        }
        log_path = run_dir / 'logs' / 'hf_upload_error.log'
        try:
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open('a', encoding='utf-8') as log:
                log.write(redact(json.dumps(diagnostic, ensure_ascii=False)) + '\n')
                log.write(redact(''.join(traceback.format_exception(
                    type(error), error, error.__traceback__))) + '\n')
            print('Log lỗi upload:', log_path)
        except OSError:
            print('Không ghi được file log; thông báo lỗi đã in ở trên.')
        return status, detail

    api = HfApi(token=token)  # Không login(), không lưu token vào file hay biến môi trường.
    try:
        api.repo_info(repo_id=HF_REPO_ID, repo_type='dataset')
    except Exception as error:
        status, detail = report_upload_error(error, 'repo_info')
        raise RuntimeError(f'Không truy cập được dataset {HF_REPO_ID} (HTTP {status or "?"}). '
                           f'\nLỗi gốc: {detail}\n'
                           'Kiểm tra Internet, repo đích và quyền của HF_TOKEN.') from None

    file_records = []
    for local_path, remote_path in files:
        digest = hashlib.sha256()
        with local_path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        file_records.append({'path_in_repo': remote_path, 'bytes': local_path.stat().st_size,
                             'sha256': digest.hexdigest()})
    upload_manifest = {
        'format_version': 1,
        'source_notebook': 'crawl_corpus_kaggle_hf_upload.ipynb',
        'repo_id': HF_REPO_ID, 'run_name': run_name,
        'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
        'dataset_revision': manifest['dataset_revision'],
        'source_sha256': manifest['source_sha256'],
        'complete': True, 'corpus_policy': 'accepted_only',
        'counts': {key: manifest[key] for key in ('selected', 'completed', 'pending', 'accepted')},
        'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
        'files': file_records,
    }
    upload_manifest_path = run_dir / 'hf_upload_manifest.json'
    upload_manifest_path.write_text(json.dumps(upload_manifest, ensure_ascii=False, indent=2) + '\n',
                                    encoding='utf-8')
    files.append((upload_manifest_path, f'manifests/{run_name}/upload_manifest.json'))
    print(f'Dataset đích: {HF_REPO_ID} | Run: {run_name} | Accepted: {manifest["accepted"]}')
    for local_path, remote_path in files:
        print(f'  {remote_path} ({local_path.stat().st_size / 1024**2:.2f} MiB)')

    for attempt in range(1, HF_UPLOAD_ATTEMPTS + 1):
        try:
            # Một commit cho corpus, manifest và ZIP của cùng run; chỉ thêm/cập nhật các path này.
            # Tạo lại operations khi retry vì SDK có thể thay đổi chúng trong quá trình upload.
            commit = api.create_commit(
                repo_id=HF_REPO_ID, repo_type='dataset',
                operations=[CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
                            for local, remote in files],
                commit_message=f'Upload crawl {run_name} ({manifest["accepted"]} accepted)',
            )
            break
        except Exception as error:
            status, detail = report_upload_error(error, 'create_commit', attempt)
            retryable = status is None or status >= 500 or status in (408, 409, 429)
            if not retryable or attempt == HF_UPLOAD_ATTEMPTS:
                raise RuntimeError(f'Upload chưa thành công (HTTP {status or "?"}, '
                                   f'{type(error).__name__}).\nLỗi gốc: {detail}\n'
                                   'Xem logs/hf_upload_error.log để biết chi tiết; '
                                   'output local vẫn còn, có thể chạy lại riêng mục 10.') from None
            print(f'Upload thất bại (lượt {attempt}/{HF_UPLOAD_ATTEMPTS}); đang thử lại...')
            time.sleep(min(2 ** attempt, 30))

    result = {
        'status': 'uploaded', 'repo_id': HF_REPO_ID, 'run_name': run_name,
        'commit_url': commit.commit_url, 'commit_oid': commit.oid,
        'files': [remote for _, remote in files],
    }
    (run_dir / 'hf_upload_result.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Upload thành công:', commit.commit_url)
    for _, remote in files:
        print(f'https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/{quote(remote, safe="/")}')
    return result


hf_upload_result = upload_current_run_to_hf()
